In [ ]:
import pandas as pd

# ============================================================
# FILES
# ============================================================

PATTERN_FILE = "new_patterns.txt"
TRANSACTION_FILE = "new_transactions.csv"
OUTPUT_FILE = "remaining_fraud_transactions.csv"


# ============================================================
# 1. READ new_transactions.csv
# ============================================================

df = pd.read_csv(TRANSACTION_FILE)

print(f"Total transactions: {len(df):,}")


# Normalize fields
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

df["from_account"] = (
    df["from_account"]
    .astype(str)
    .str.strip()
)

df["to_account"] = (
    df["to_account"]
    .astype(str)
    .str.strip()
)

df["amount_paid"] = pd.to_numeric(
    df["amount_paid"],
    errors="coerce"
)

df["payment_mode"] = (
    df["payment_mode"]
    .astype(str)
    .str.strip()
)


# ============================================================
# 2. ISOLATE ALL FRAUD / LAUNDERING TRANSACTIONS
# ============================================================

fraud_mask = (
    df["is_laundering"]
    .astype(str)
    .str.lower()
    .isin(["1", "true", "yes"])
)

fraud = df[fraud_mask].copy()

print(
    f"Total fraud/laundering transactions: "
    f"{len(fraud):,}"
)


# ============================================================
# 3. READ new_patterns.txt
# ============================================================

pattern_transactions = []

with open(
    PATTERN_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        # Ignore empty lines
        if not line:
            continue

        # Ignore BEGIN / END lines
        if line.startswith("BEGIN"):
            continue

        if line.startswith("END"):
            continue

        parts = [
            x.strip()
            for x in line.split(",")
        ]

        # Expected new_patterns format:
        #
        # 0 timestamp
        # 1 from_bank
        # 2 from_account
        # 3 to_bank
        # 4 to_account
        # 5 amount
        # 6 payment_mode
        # 7 is_laundering
        # 8 from_district
        # 9 to_district

        if len(parts) < 10:
            continue

        pattern_transactions.append({
            "timestamp": pd.to_datetime(
                parts[0],
                errors="coerce"
            ),
            "from_account": parts[2],
            "to_account": parts[4],
            "amount": pd.to_numeric(
                parts[5],
                errors="coerce"
            ),
            "payment_mode": parts[6]
        })


patterns_df = pd.DataFrame(
    pattern_transactions
)

print(
    f"Transactions in new_patterns.txt: "
    f"{len(patterns_df):,}"
)


# ============================================================
# 4. CREATE MATCHING KEYS
# ============================================================
#
# We match using:
#
#   timestamp
#   from_account
#   to_account
#   amount
#   payment_mode
#
# We do NOT need bank/district for matching because
# new_patterns.txt already contains the corresponding
# transaction from the new dataset.
#
# ============================================================

fraud["_match_key"] = list(zip(
    fraud["timestamp"],
    fraud["from_account"],
    fraud["to_account"],
    fraud["amount_paid"].round(2),
    fraud["payment_mode"].str.lower()
))


patterns_df["_match_key"] = list(zip(
    patterns_df["timestamp"],
    patterns_df["from_account"],
    patterns_df["to_account"],
    patterns_df["amount"].round(2),
    patterns_df["payment_mode"].str.lower()
))


# ============================================================
# 5. FIND TRANSACTIONS ALREADY IN new_patterns.txt
# ============================================================

pattern_keys = set(
    patterns_df["_match_key"]
)


# ============================================================
# 6. ISOLATE REMAINING FRAUD
# ============================================================

remaining_fraud = fraud[
    ~fraud["_match_key"].isin(pattern_keys)
].copy()


# Remove temporary column
remaining_fraud = remaining_fraud.drop(
    columns=["_match_key"]
)


# ============================================================
# 7. SAVE
# ============================================================

remaining_fraud.to_csv(
    OUTPUT_FILE,
    index=False
)


# ============================================================
# 8. RESULTS
# ============================================================

print("\n" + "=" * 70)
print("RESULT")
print("=" * 70)

print(
    f"Total transactions                  : {len(df):,}"
)

print(
    f"Total fraud/laundering transactions : {len(fraud):,}"
)

print(
    f"Transactions represented in "
    f"new_patterns.txt                   : {len(patterns_df):,}"
)

print(
    f"Remaining unexplained fraud         : "
    f"{len(remaining_fraud):,}"
)

print(
    f"\nSaved to: {OUTPUT_FILE}"
)


# ============================================================
# 9. PREVIEW
# ============================================================

display(
    remaining_fraud.head(20)
)

In [ ]:
# ============================================================
# EXCLUSIVE UNKNOWN PATTERN DISCOVERY
# ============================================================
#
# IMPORTANT:
# A transaction can belong to ONLY ONE discovered pattern.
#
# Algorithm:
#
#   1. Sort fraud transactions chronologically
#   2. Pick earliest unused transaction
#   3. Try to construct a pattern starting from that transaction
#   4. If pattern found:
#        - save pattern
#        - mark ALL its transactions as used
#   5. If no pattern found:
#        - mark seed as processed/unpatterned
#   6. Move to next unused transaction
#
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

# INPUT_FILE = "remaining_fraud_transactions.csv"
INPUT_FILE = "still_unpatterned_fraud.csv"

OUTPUT_FILE = "unknown_patterns.txt"
SUMMARY_FILE = "unknown_pattern_summary.csv"
UNPATTERNED_FILE = "still_unpatterned_fraud.csv"

# Temporal constraints
FAN_WINDOW_MINUTES = 1200
CHAIN_WINDOW_MINUTES = 1200

# Structural constraints
MIN_FAN_DEGREE = 2

MIN_CHAIN_LENGTH = 2
MAX_CHAIN_LENGTH = 500

# Amount similarity for chaining
AMOUNT_TOLERANCE = 50.0


# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("=" * 80)
print("LOADING FRAUD TRANSACTIONS")
print("=" * 80)

print(f"Input transactions: {len(df):,}")

# ------------------------------------------------------------
# NORMALIZE
# ------------------------------------------------------------

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

df["from_account"] = (
    df["from_account"]
    .astype(str)
    .str.strip()
)

df["to_account"] = (
    df["to_account"]
    .astype(str)
    .str.strip()
)

df["amount_paid"] = pd.to_numeric(
    df["amount_paid"],
    errors="coerce"
)

df["payment_mode"] = (
    df["payment_mode"]
    .astype(str)
    .str.strip()
)

df["from_district"] = (
    df["from_district"]
    .astype(str)
    .str.strip()
)

df["to_district"] = (
    df["to_district"]
    .astype(str)
    .str.strip()
)

df["from_indian_bank"] = (
    df["from_indian_bank"]
    .astype(str)
    .str.strip()
)

df["to_indian_bank"] = (
    df["to_indian_bank"]
    .astype(str)
    .str.strip()
)

df = df.dropna(
    subset=[
        "timestamp",
        "from_account",
        "to_account",
        "amount_paid"
    ]
).copy()

# ------------------------------------------------------------
# UNIQUE TRANSACTION ID
# ------------------------------------------------------------

df = df.reset_index(drop=True)

df["transaction_id"] = df.index

# Chronological order
df = df.sort_values(
    "timestamp"
).reset_index(drop=True)

print(f"Valid transactions: {len(df):,}")


# ------------------------------------------------------------
# TRANSACTION FORMATTER
# ------------------------------------------------------------

def format_transaction(row):

    return ",".join([
        row["timestamp"].strftime("%Y/%m/%d %H:%M"),
        str(row["from_indian_bank"]),
        str(row["from_account"]),
        str(row["to_indian_bank"]),
        str(row["to_account"]),
        f'{row["amount_paid"]:.2f}',
        str(row["payment_mode"]),
        str(row["is_laundering"]),
        str(row["from_district"]),
        str(row["to_district"])
    ])


# ------------------------------------------------------------
# AMOUNT COMPATIBILITY
# ------------------------------------------------------------

def amount_compatible(a, b):

    if a <= 0 or b <= 0:
        return True

    ratio = b / a

    return (
        1 - AMOUNT_TOLERANCE
        <= ratio
        <=
        1 + AMOUNT_TOLERANCE
    )


# ------------------------------------------------------------
# BUILD INDEXES
# ------------------------------------------------------------

outgoing = defaultdict(list)
incoming = defaultdict(list)

for idx, row in df.iterrows():

    outgoing[
        row["from_account"]
    ].append(idx)

    incoming[
        row["to_account"]
    ].append(idx)


# ------------------------------------------------------------
# IMPORTANT:
# USED TRANSACTION SET
# ------------------------------------------------------------

used_transactions = set()

# Transactions that were examined but didn't form a pattern
unpatterned_transactions = set()

# Final discovered patterns
discovered_patterns = []


# ============================================================
# PATTERN DETECTORS
# ============================================================

# Each detector receives:
#
#   seed_idx
#   available transactions
#
# and returns:
#
#   {
#       "type": ...,
#       "description": ...,
#       "transactions": [...]
#   }
#
# OR None
#
# The detectors ONLY use transactions that are currently
# unused.
# ============================================================


# ------------------------------------------------------------
# 1. FAN-IN
# ------------------------------------------------------------

def find_fan_in(seed_idx, available):

    seed = df.loc[seed_idx]

    target = seed["to_account"]

    # All currently available incoming transactions
    candidates = [
        i for i in incoming.get(target, [])
        if i in available
    ]

    if len(candidates) < MIN_FAN_DEGREE:
        return None

    seed_time = seed["timestamp"]

    valid = []

    for idx in candidates:

        row = df.loc[idx]

        delta = (
            row["timestamp"] - seed_time
        ).total_seconds() / 60

        if abs(delta) <= FAN_WINDOW_MINUTES:

            valid.append(idx)

    # Distinct source accounts
    source_accounts = set(
        df.loc[
            valid,
            "from_account"
        ]
    )

    if len(source_accounts) < MIN_FAN_DEGREE:
        return None

    # Keep chronological transactions
    valid = sorted(
        valid,
        key=lambda i: df.loc[
            i,
            "timestamp"
        ]
    )

    return {
        "type": "FAN-IN",
        "description": (
            f"{len(source_accounts)}-degree Fan-In "
            f"to {target}"
        ),
        "transactions": valid
    }


# ------------------------------------------------------------
# 2. FAN-OUT
# ------------------------------------------------------------

def find_fan_out(seed_idx, available):

    seed = df.loc[seed_idx]

    source = seed["from_account"]

    candidates = [
        i for i in outgoing.get(source, [])
        if i in available
    ]

    if len(candidates) < MIN_FAN_DEGREE:
        return None

    seed_time = seed["timestamp"]

    valid = []

    for idx in candidates:

        row = df.loc[idx]

        delta = (
            row["timestamp"] - seed_time
        ).total_seconds() / 60

        if abs(delta) <= FAN_WINDOW_MINUTES:

            valid.append(idx)

    destination_accounts = set(
        df.loc[
            valid,
            "to_account"
        ]
    )

    if len(destination_accounts) < MIN_FAN_DEGREE:
        return None

    valid = sorted(
        valid,
        key=lambda i: df.loc[
            i,
            "timestamp"
        ]
    )

    return {
        "type": "FAN-OUT",
        "description": (
            f"{len(destination_accounts)}-degree Fan-Out "
            f"from {source}"
        ),
        "transactions": valid
    }


# ------------------------------------------------------------
# 3. GATHER-SCATTER
#
# many -> HUB -> many
# ------------------------------------------------------------

def find_gather_scatter(seed_idx, available):

    seed = df.loc[seed_idx]

    hub = seed["to_account"]

    # Incoming
    in_candidates = [
        i for i in incoming.get(hub, [])
        if i in available
    ]

    if len(in_candidates) < MIN_FAN_DEGREE:
        return None

    seed_time = seed["timestamp"]

    incoming_valid = [
        i for i in in_candidates
        if abs(
            (
                df.loc[i, "timestamp"]
                - seed_time
            ).total_seconds() / 60
        ) <= CHAIN_WINDOW_MINUTES
    ]

    source_accounts = set(
        df.loc[
            incoming_valid,
            "from_account"
        ]
    )

    if len(source_accounts) < MIN_FAN_DEGREE:
        return None

    # Outgoing must happen after incoming
    latest_incoming = max(
        df.loc[
            incoming_valid,
            "timestamp"
        ]
    )

    out_candidates = [
        i for i in outgoing.get(hub, [])
        if i in available
    ]

    outgoing_valid = [
        i for i in out_candidates
        if (
            df.loc[i, "timestamp"] >= latest_incoming
            and
            (
                df.loc[i, "timestamp"]
                - latest_incoming
            ).total_seconds() / 60
            <= CHAIN_WINDOW_MINUTES
        )
    ]

    destination_accounts = set(
        df.loc[
            outgoing_valid,
            "to_account"
        ]
    )

    if len(destination_accounts) < MIN_FAN_DEGREE:
        return None

    transactions = sorted(
        incoming_valid + outgoing_valid,
        key=lambda i: df.loc[
            i,
            "timestamp"
        ]
    )

    return {
        "type": "GATHER-SCATTER",
        "description": (
            f"Gather-Scatter: "
            f"{len(source_accounts)} sources -> "
            f"{hub} -> "
            f"{len(destination_accounts)} destinations"
        ),
        "transactions": transactions
    }


# ------------------------------------------------------------
# 4. SCATTER-GATHER
#
# source -> many intermediates -> destination
# ------------------------------------------------------------

def find_scatter_gather(seed_idx, available):

    seed = df.loc[seed_idx]

    source = seed["from_account"]

    first_hops = [
        i for i in outgoing.get(source, [])
        if i in available
    ]

    seed_time = seed["timestamp"]

    first_hops = [
        i for i in first_hops
        if abs(
            (
                df.loc[i, "timestamp"]
                - seed_time
            ).total_seconds() / 60
        ) <= CHAIN_WINDOW_MINUTES
    ]

    intermediates = set(
        df.loc[
            first_hops,
            "to_account"
        ]
    )

    if len(intermediates) < MIN_FAN_DEGREE:
        return None

    # Find common destination
    destination_map = defaultdict(list)

    for intermediary in intermediates:

        for idx in outgoing.get(
            intermediary,
            []
        ):

            if idx not in available:
                continue

            row = df.loc[idx]

            if row["timestamp"] < seed_time:
                continue

            if (
                row["timestamp"]
                - seed_time
            ).total_seconds() / 60 > CHAIN_WINDOW_MINUTES:
                break

            destination_map[
                row["to_account"]
            ].append(idx)

    best_destination = None
    best_transactions = []

    for destination, indices in destination_map.items():

        sending_intermediates = set(
            df.loc[
                indices,
                "from_account"
            ]
        )

        if len(sending_intermediates) >= MIN_FAN_DEGREE:

            if len(indices) > len(best_transactions):

                best_destination = destination
                best_transactions = indices

    if best_destination is None:
        return None

    transactions = sorted(
        first_hops + best_transactions,
        key=lambda i: df.loc[
            i,
            "timestamp"
        ]
    )

    return {
        "type": "SCATTER-GATHER",
        "description": (
            f"Scatter-Gather: "
            f"{source} -> "
            f"{len(intermediates)} intermediates -> "
            f"{best_destination}"
        ),
        "transactions": transactions
    }


# ------------------------------------------------------------
# 5. MULTI-HOP CHAIN
#
# A -> B -> C -> D
# ------------------------------------------------------------

def find_chain(seed_idx, available):

    seed = df.loc[seed_idx]

    chain = [seed_idx]

    visited = {
        seed["from_account"],
        seed["to_account"]
    }

    current_idx = seed_idx

    while len(chain) < MAX_CHAIN_LENGTH:

        current = df.loc[current_idx]

        next_account = current["to_account"]

        candidates = [
            i for i in outgoing.get(
                next_account,
                []
            )
            if i in available
        ]

        candidates = [
            i for i in candidates
            if i not in chain
        ]

        best = None

        for idx in candidates:

            row = df.loc[idx]

            if row["timestamp"] <= current["timestamp"]:
                continue

            delta = (
                row["timestamp"]
                - current["timestamp"]
            ).total_seconds() / 60

            if delta > CHAIN_WINDOW_MINUTES:
                break

            if row["to_account"] in visited:
                continue

            if not amount_compatible(
                current["amount_paid"],
                row["amount_paid"]
            ):
                continue

            best = idx
            break

        if best is None:
            break

        chain.append(best)

        visited.add(
            df.loc[
                best,
                "to_account"
            ]
        )

        current_idx = best

    if len(chain) < MIN_CHAIN_LENGTH:
        return None

    accounts = [
        df.loc[
            chain[0],
            "from_account"
        ]
    ]

    for idx in chain:

        accounts.append(
            df.loc[
                idx,
                "to_account"
            ]
        )

    return {
        "type": "MULTI-HOP CHAIN",
        "description": (
            f"{len(chain)}-transaction chain: "
            f"{' -> '.join(accounts)}"
        ),
        "transactions": chain
    }


# ------------------------------------------------------------
# 6. CYCLE
#
# A -> B -> C -> A
# ------------------------------------------------------------

def find_cycle(seed_idx, available):

    seed = df.loc[seed_idx]

    start = seed["from_account"]

    current_account = seed["to_account"]

    cycle = [seed_idx]

    visited = {
        start,
        current_account
    }

    current_time = seed["timestamp"]

    while len(cycle) < 5:

        candidates = [
            i for i in outgoing.get(
                current_account,
                []
            )
            if i in available
        ]

        next_idx = None

        for idx in candidates:

            row = df.loc[idx]

            if row["timestamp"] <= current_time:
                continue

            delta = (
                row["timestamp"]
                - current_time
            ).total_seconds() / 60

            if delta > CHAIN_WINDOW_MINUTES:
                break

            destination = row["to_account"]

            # Closing cycle
            if destination == start:

                if len(cycle) >= 2:

                    cycle.append(idx)

                    return {
                        "type": "CYCLE",
                        "description": (
                            f"Cycle detected returning to "
                            f"{start}"
                        ),
                        "transactions": cycle
                    }

            if destination in visited:
                continue

            next_idx = idx
            break

        if next_idx is None:
            break

        cycle.append(next_idx)

        current_account = df.loc[
            next_idx,
            "to_account"
        ]

        current_time = df.loc[
            next_idx,
            "timestamp"
        ]

        visited.add(
            current_account
        )

    return None


# ============================================================
# PATTERN PRIORITY
# ============================================================
#
# More specific structures first.
#
# This is important.
#
# Example:
#
# A -> B
# A -> C
# A -> D
# D -> X
#
# Could be interpreted as Fan-Out OR a more complex structure.
#
# We prefer the more informative structure.
# ============================================================

PATTERN_DETECTORS = [
    ("GATHER-SCATTER", find_gather_scatter),
    ("SCATTER-GATHER", find_scatter_gather),
    ("CYCLE", find_cycle),
    ("FAN-IN", find_fan_in),
    ("FAN-OUT", find_fan_out),
    ("MULTI-HOP CHAIN", find_chain),
]


# ============================================================
# MAIN EXCLUSIVE DISCOVERY LOOP
# ============================================================

print("\n")
print("=" * 80)
print("STARTING EXCLUSIVE PATTERN DISCOVERY")
print("=" * 80)

total = len(df)

for seed_idx in range(total):

    # Already consumed by a previous pattern
    if seed_idx in used_transactions:
        continue

    # Already checked and found nothing
    if seed_idx in unpatterned_transactions:
        continue

    available = set(
        range(total)
    ) - used_transactions

    found_pattern = None

    # --------------------------------------------------------
    # Try every pattern against THIS seed
    # --------------------------------------------------------

    for pattern_name, detector in PATTERN_DETECTORS:

        result = detector(
            seed_idx,
            available
        )

        if result is None:
            continue

        # ----------------------------------------------------
        # Make sure the pattern really contains the seed
        # ----------------------------------------------------

        if seed_idx not in result["transactions"]:
            continue

        # ----------------------------------------------------
        # Safety:
        # no transaction may already be consumed
        # ----------------------------------------------------

        if any(
            idx in used_transactions
            for idx in result["transactions"]
        ):
            continue

        found_pattern = result

        break

    # --------------------------------------------------------
    # Pattern found
    # --------------------------------------------------------

    if found_pattern is not None:

        transactions = found_pattern[
            "transactions"
        ]

        used_transactions.update(
            transactions
        )

        discovered_patterns.append(
            found_pattern
        )

        print(
            f"[PATTERN #{len(discovered_patterns):04d}] "
            f"{found_pattern['type']:18s} | "
            f"{len(transactions):2d} transactions | "
            f"{found_pattern['description']}"
        )

    # --------------------------------------------------------
    # No pattern found
    # --------------------------------------------------------

    else:

        unpatterned_transactions.add(
            seed_idx
        )


# ============================================================
# WRITE UNKNOWN PATTERNS
# ============================================================

print("\n")
print("=" * 80)
print("WRITING unknown_patterns.txt")
print("=" * 80)

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    for number, pattern in enumerate(
        discovered_patterns,
        start=1
    ):

        pattern_type = pattern["type"]
        description = pattern["description"]

        f.write(
            f"BEGIN LAUNDERING ATTEMPT - "
            f"{pattern_type}: {description}\n"
        )

        rows = df.loc[
            pattern["transactions"]
        ].sort_values(
            "timestamp"
        )

        for _, row in rows.iterrows():

            f.write(
                format_transaction(row)
                + "\n"
            )

        f.write(
            f"END LAUNDERING ATTEMPT - "
            f"{pattern_type}: {description}\n\n"
        )


# ============================================================
# SUMMARY
# ============================================================

summary = []

for pattern_type in sorted(
    set(
        p["type"]
        for p in discovered_patterns
    )
):

    matching = [
        p for p in discovered_patterns
        if p["type"] == pattern_type
    ]

    transaction_ids = set()

    for p in matching:

        transaction_ids.update(
            p["transactions"]
        )

    summary.append({
        "pattern_type": pattern_type,
        "pattern_instances": len(matching),
        "transactions_used": len(
            transaction_ids
        )
    })


summary_df = pd.DataFrame(summary)

if len(summary_df) > 0:

    summary_df = summary_df.sort_values(
        "pattern_instances",
        ascending=False
    )

else:

    summary_df = pd.DataFrame(
        columns=[
            "pattern_type",
            "pattern_instances",
            "transactions_used"
        ]
    )

summary_df.to_csv(
    SUMMARY_FILE,
    index=False
)


# ============================================================
# SAVE STILL-UNPATTERNED TRANSACTIONS
# ============================================================

still_unpatterned = df.loc[
    sorted(unpatterned_transactions)
].copy()

still_unpatterned.to_csv(
    UNPATTERNED_FILE,
    index=False
)


# ============================================================
# VALIDATION
# ============================================================

patterned_count = len(
    used_transactions
)

unpatterned_count = len(
    unpatterned_transactions
)

print("\n")
print("=" * 80)
print("DISCOVERY COMPLETE")
print("=" * 80)

print(
    f"Total fraud transactions : {total:,}"
)

print(
    f"Transactions in patterns : {patterned_count:,}"
)

print(
    f"Still unpatterned         : {unpatterned_count:,}"
)

print(
    f"Discovered pattern count  : "
    f"{len(discovered_patterns):,}"
)

print(
    f"\nValidation: "
    f"{patterned_count + unpatterned_count:,} "
    f"/ {total:,}"
)

print("\nPattern summary:")
display(summary_df)

print("\nOutput files:")
print(f"  ✓ {OUTPUT_FILE}")
print(f"  ✓ {SUMMARY_FILE}")
print(f"  ✓ {UNPATTERNED_FILE}")

In [ ]:
# ============================================================
# ACCOUNT REUSE / DISTRIBUTION ANALYSIS
# remaining_fraud_transactions.csv
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# INPUT_FILE = "remaining_fraud_transactions.csv"
# INPUT_FILE = "still_unpatterned_fraud.csv"
INPUT_FILE = "fraud.csv"

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

df["from_account"] = (
    df["from_account"]
    .astype(str)
    .str.strip()
)

df["to_account"] = (
    df["to_account"]
    .astype(str)
    .str.strip()
)

df["amount_paid"] = pd.to_numeric(
    df["amount_paid"],
    errors="coerce"
)

print("=" * 70)
print("ACCOUNT REUSE ANALYSIS")
print("=" * 70)

print(f"Total fraud transactions : {len(df):,}")

# ------------------------------------------------------------
# 2. ALL UNIQUE ACCOUNTS
# ------------------------------------------------------------

all_accounts = set(
    df["from_account"]
).union(
    set(df["to_account"])
)

print(f"Unique accounts involved : {len(all_accounts):,}")


# ------------------------------------------------------------
# 3. TRANSACTION COUNT PER ACCOUNT
#
# Count BOTH:
#   outgoing transactions
#   incoming transactions
#   total involvement
# ------------------------------------------------------------

outgoing_count = (
    df.groupby("from_account")
    .size()
    .rename("outgoing_transactions")
)

incoming_count = (
    df.groupby("to_account")
    .size()
    .rename("incoming_transactions")
)

account_stats = pd.DataFrame(
    index=pd.Index(
        list(all_accounts),
        name="account"
    )
)

account_stats = account_stats.join(
    outgoing_count,
    how="left"
)

account_stats = account_stats.join(
    incoming_count,
    how="left"
)

account_stats = account_stats.fillna(0)

account_stats["outgoing_transactions"] = (
    account_stats["outgoing_transactions"]
    .astype(int)
)

account_stats["incoming_transactions"] = (
    account_stats["incoming_transactions"]
    .astype(int)
)

account_stats["total_transactions"] = (
    account_stats["outgoing_transactions"]
    +
    account_stats["incoming_transactions"]
)

# ------------------------------------------------------------
# 4. CLASSIFY ACCOUNT REUSE
# ------------------------------------------------------------

account_stats["account_type"] = np.select(
    [
        (
            (account_stats["outgoing_transactions"] > 0) &
            (account_stats["incoming_transactions"] > 0)
        ),
        (
            account_stats["outgoing_transactions"] > 0
        ),
        (
            account_stats["incoming_transactions"] > 0
        )
    ],
    [
        "Both incoming & outgoing",
        "Outgoing only",
        "Incoming only"
    ],
    default="None"
)

account_stats["reuse_category"] = np.select(
    [
        account_stats["total_transactions"] == 1,
        account_stats["total_transactions"] == 2,
        account_stats["total_transactions"].between(3, 5),
        account_stats["total_transactions"].between(6, 10),
        account_stats["total_transactions"] > 10
    ],
    [
        "1 transaction",
        "2 transactions",
        "3–5 transactions",
        "6–10 transactions",
        ">10 transactions"
    ],
    default="Other"
)


# ------------------------------------------------------------
# 5. PRINT ACCOUNT REUSE STATISTICS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ACCOUNT REUSE")
print("=" * 70)

reuse_distribution = (
    account_stats["reuse_category"]
    .value_counts()
)

print("\nAccounts by number of transaction involvements:")
print(reuse_distribution)


# ------------------------------------------------------------
# 6. HOW MANY TRANSACTIONS COME FROM REUSED ACCOUNTS?
# ------------------------------------------------------------

transaction_account_frequency = (
    account_stats["total_transactions"]
)

# Map frequency to source account
df["from_account_frequency"] = (
    df["from_account"]
    .map(transaction_account_frequency)
)

df["to_account_frequency"] = (
    df["to_account"]
    .map(transaction_account_frequency)
)

# A transaction is associated with a reused account
# if either endpoint participates in >1 transaction.
df["has_reused_account"] = (
    (df["from_account_frequency"] > 1) |
    (df["to_account_frequency"] > 1)
)

reused_transaction_count = (
    df["has_reused_account"].sum()
)

unique_transaction_count = (
    len(df) - reused_transaction_count
)

print("\nTransaction-level result:")
print(
    f"Transactions involving ONLY one-time accounts : "
    f"{unique_transaction_count:,}"
)

print(
    f"Transactions involving a reused account        : "
    f"{reused_transaction_count:,}"
)

print(
    f"Percentage involving reused account            : "
    f"{reused_transaction_count / len(df) * 100:.2f}%"
)


# ============================================================
# VISUALIZATION 1
# Account reuse distribution
# ============================================================

plt.figure(figsize=(10, 6))

order = [
    "1 transaction",
    "2 transactions",
    "3–5 transactions",
    "6–10 transactions",
    ">10 transactions"
]

values = [
    reuse_distribution.get(x, 0)
    for x in order
]

plt.bar(order, values)

plt.title(
    "Distribution of Fraud Accounts by Transaction Involvement"
)

plt.xlabel(
    "Number of transactions involving account"
)

plt.ylabel(
    "Number of accounts"
)

plt.xticks(rotation=20)

plt.tight_layout()
plt.show()


# ============================================================
# VISUALIZATION 2
# Histogram of transaction involvement per account
# ============================================================

plt.figure(figsize=(10, 6))

plt.hist(
    account_stats["total_transactions"],
    bins=30
)

plt.title(
    "Transaction Involvement Distribution per Account"
)

plt.xlabel(
    "Number of transactions involving account"
)

plt.ylabel(
    "Number of accounts"
)

plt.tight_layout()
plt.show()


# ============================================================
# VISUALIZATION 3
# Incoming vs outgoing transactions
# ============================================================

plt.figure(figsize=(10, 6))

plt.scatter(
    account_stats["incoming_transactions"],
    account_stats["outgoing_transactions"],
    alpha=0.6
)

plt.xlabel(
    "Incoming transactions"
)

plt.ylabel(
    "Outgoing transactions"
)

plt.title(
    "Account Activity: Incoming vs Outgoing Transactions"
)

plt.tight_layout()
plt.show()


# ============================================================
# 7. TOP REUSED ACCOUNTS
# ============================================================

print("\n" + "=" * 70)
print("TOP REUSED ACCOUNTS")
print("=" * 70)

top_accounts = (
    account_stats
    .sort_values(
        "total_transactions",
        ascending=False
    )
    .head(30)
)

display(
    top_accounts.reset_index()
)


# ============================================================
# VISUALIZATION 4
# Top 20 most reused accounts
# ============================================================

top20 = (
    account_stats
    .sort_values(
        "total_transactions",
        ascending=False
    )
    .head(20)
    .sort_values(
        "total_transactions"
    )
)

plt.figure(figsize=(10, 8))

plt.barh(
    top20.index.astype(str),
    top20["total_transactions"]
)

plt.title(
    "Top 20 Most Reused Accounts"
)

plt.xlabel(
    "Total transaction involvement"
)

plt.ylabel(
    "Account"
)

plt.tight_layout()
plt.show()


# ============================================================
# 8. ACCOUNT-PAIR REUSE
#
# Does the SAME:
#
#       A -> B
#
# happen repeatedly?
# ============================================================

pair_counts = (
    df.groupby(
        [
            "from_account",
            "to_account"
        ]
    )
    .size()
    .reset_index(
        name="transaction_count"
    )
    .sort_values(
        "transaction_count",
        ascending=False
    )
)

print("\n" + "=" * 70)
print("ACCOUNT-PAIR REUSE")
print("=" * 70)

print(
    f"Unique account pairs : {len(pair_counts):,}"
)

print("\nTop repeated account pairs:")

display(
    pair_counts.head(30)
)


# ============================================================
# VISUALIZATION 5
# Distribution of repeated account pairs
# ============================================================

pair_distribution = (
    pair_counts["transaction_count"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(10, 6))

plt.bar(
    pair_distribution.index,
    pair_distribution.values
)

plt.title(
    "Distribution of Transactions per Account Pair"
)

plt.xlabel(
    "Number of transactions for same A → B pair"
)

plt.ylabel(
    "Number of account pairs"
)

plt.tight_layout()
plt.show()


# ============================================================
# 9. HOW MANY TRANSACTIONS ARE PART OF REPEATED PAIRS?
# ============================================================

repeated_pairs = pair_counts[
    pair_counts["transaction_count"] > 1
]

transactions_in_repeated_pairs = (
    repeated_pairs["transaction_count"]
    .sum()
)

print("\n" + "=" * 70)
print("REPEATED PAIR ANALYSIS")
print("=" * 70)

print(
    f"Repeated account pairs       : "
    f"{len(repeated_pairs):,}"
)

print(
    f"Transactions in repeated pairs: "
    f"{transactions_in_repeated_pairs:,}"
)

print(
    f"Percentage of fraud transactions: "
    f"{transactions_in_repeated_pairs / len(df) * 100:.2f}%"
)


# ============================================================
# 10. REPEATED SOURCE ACCOUNTS
# ============================================================

source_reuse = (
    df.groupby("from_account")
    .size()
    .reset_index(
        name="outgoing_count"
    )
    .sort_values(
        "outgoing_count",
        ascending=False
    )
)

print("\n" + "=" * 70)
print("SOURCE ACCOUNT REUSE")
print("=" * 70)

display(
    source_reuse.head(30)
)


# ============================================================
# 11. REPEATED DESTINATION ACCOUNTS
# ============================================================

destination_reuse = (
    df.groupby("to_account")
    .size()
    .reset_index(
        name="incoming_count"
    )
    .sort_values(
        "incoming_count",
        ascending=False
    )
)

print("\n" + "=" * 70)
print("DESTINATION ACCOUNT REUSE")
print("=" * 70)

display(
    destination_reuse.head(30)
)


# ============================================================
# 12. SAVE ACCOUNT ANALYSIS
# ============================================================

account_stats.to_csv(
    "fraud_account_statistics.csv"
)

pair_counts.to_csv(
    "fraud_account_pair_statistics.csv",
    index=False
)

print("\nSaved:")
print("  ✓ fraud_account_statistics.csv")
print("  ✓ fraud_account_pair_statistics.csv")

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# LOAD
# ============================================================

df = pd.read_csv("fraud.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

df["from_account"] = df["from_account"].astype(str).str.strip()
df["to_account"] = df["to_account"].astype(str).str.strip()

df["amount_paid"] = pd.to_numeric(
    df["amount_paid"], errors="coerce"
)

df = df.sort_values("timestamp").reset_index(drop=True)

# Give every transaction an ID
df["transaction_id"] = df.index


# ============================================================
# ACCOUNT FREQUENCY
# ============================================================

out_counts = df["from_account"].value_counts()
in_counts = df["to_account"].value_counts()

account_total = (
    out_counts.add(in_counts, fill_value=0)
    .sort_values(ascending=False)
)

reused_accounts = set(
    account_total[account_total > 1].index
)

print("=" * 70)
print("REUSED ACCOUNT DIAGNOSTIC")
print("=" * 70)

print("Reused accounts:", len(reused_accounts))


# ============================================================
# GET TRANSACTIONS INVOLVING REUSED ACCOUNTS
# ============================================================

reused_tx = df[
    df["from_account"].isin(reused_accounts) |
    df["to_account"].isin(reused_accounts)
].copy()

print("Transactions involving reused accounts:", len(reused_tx))


# ============================================================
# BUILD ACCOUNT -> TRANSACTIONS
# ============================================================

account_transactions = {}

for account in reused_accounts:

    tx = df[
        (df["from_account"] == account) |
        (df["to_account"] == account)
    ].copy()

    account_transactions[account] = tx


# ============================================================
# CLASSIFY EACH REUSED ACCOUNT
# ============================================================

diagnostics = []

for account in sorted(reused_accounts):

    tx = account_transactions[account].sort_values("timestamp")

    incoming = tx[tx["to_account"] == account]
    outgoing = tx[tx["from_account"] == account]

    # ----------------------------------------
    # FAN-IN
    # ----------------------------------------

    incoming_sources = set(incoming["from_account"])

    is_fan_in = len(incoming_sources) >= 2


    # ----------------------------------------
    # FAN-OUT
    # ----------------------------------------

    outgoing_destinations = set(outgoing["to_account"])

    is_fan_out = len(outgoing_destinations) >= 2


    # ----------------------------------------
    # CHAIN
    # ----------------------------------------

    is_chain = (
        len(incoming) > 0 and
        len(outgoing) > 0
    )


    # ----------------------------------------
    # Check amount compatibility
    # ----------------------------------------

    amount_compatible = False
    min_amount_difference = None

    if len(incoming) > 0 and len(outgoing) > 0:

        differences = []

        for _, in_tx in incoming.iterrows():

            for _, out_tx in outgoing.iterrows():

                diff = abs(
                    in_tx["amount_paid"] -
                    out_tx["amount_paid"]
                )

                differences.append(diff)

        if differences:

            min_amount_difference = min(differences)

            amount_compatible = (
                min_amount_difference <= AMOUNT_TOLERANCE
            )


    # ----------------------------------------
    # Time compatibility
    # ----------------------------------------

    time_compatible = False
    min_time_difference = None

    if len(incoming) > 0 and len(outgoing) > 0:

        time_differences = []

        for _, in_tx in incoming.iterrows():

            for _, out_tx in outgoing.iterrows():

                diff = abs(
                    (out_tx["timestamp"] -
                     in_tx["timestamp"]).total_seconds()
                    / 60
                )

                time_differences.append(diff)

        if time_differences:

            min_time_difference = min(time_differences)

            time_compatible = (
                min_time_difference <= CHAIN_WINDOW_MINUTES
            )


    # ----------------------------------------
    # Determine structural type
    # ----------------------------------------

    if is_chain and amount_compatible and time_compatible:

        structural_type = "CHAIN_CANDIDATE"

    elif is_fan_in:

        structural_type = "FAN_IN_CANDIDATE"

    elif is_fan_out:

        structural_type = "FAN_OUT_CANDIDATE"

    elif is_chain:

        structural_type = "CHAIN_BUT_AMOUNT_OR_TIME_FAILED"

    elif len(tx) == 2:

        structural_type = "REUSED_ACCOUNT_BUT_NO_PATTERN"

    else:

        structural_type = "OTHER"


    diagnostics.append({

        "account": account,

        "transaction_count": len(tx),

        "incoming_count": len(incoming),

        "outgoing_count": len(outgoing),

        "unique_incoming_sources":
            len(incoming_sources),

        "unique_outgoing_destinations":
            len(outgoing_destinations),

        "is_fan_in": is_fan_in,

        "is_fan_out": is_fan_out,

        "has_in_out_chain": is_chain,

        "amount_compatible":
            amount_compatible,

        "min_amount_difference":
            min_amount_difference,

        "time_compatible":
            time_compatible,

        "min_time_difference_minutes":
            min_time_difference,

        "classification":
            structural_type
    })


diagnostic_df = pd.DataFrame(diagnostics)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STRUCTURAL CLASSIFICATION")
print("=" * 70)

print(
    diagnostic_df["classification"]
    .value_counts()
)


print("\n" + "=" * 70)
print("DETAILED DIAGNOSTICS")
print("=" * 70)

display(
    diagnostic_df.sort_values(
        ["classification", "account"]
    )
)


# ============================================================
# SAVE
# ============================================================

diagnostic_df.to_csv(
    "reused_account_diagnostics.csv",
    index=False
)

print("\nSaved:")
print("reused_account_diagnostics.csv")

In [ ]:
import pandas as pd

# ============================================================
# LOAD ORIGINAL TRANSACTION FILE
# ============================================================

input_file = "new_transactions.csv"

df = pd.read_csv(input_file)

print("Original dataset:")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())


# ============================================================
# NORMALIZE FRAUD FLAG
# ============================================================

# Handle True/False, 1/0, "True"/"False", etc.
df["is_laundering"] = (
    df["is_laundering"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["true", "1", "yes"])
)


# ============================================================
# SELECT ONLY FRAUD TRANSACTIONS
# ============================================================

fraud_df = df[df["is_laundering"] == False].copy()


# ============================================================
# SAVE
# ============================================================

output_file = "legits.csv"

fraud_df.to_csv(
    output_file,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("FRAUD DATASET CREATED")
print("=" * 60)

print("Total transactions :", len(df))
print("Fraud transactions :", len(fraud_df))
print("Non-fraud           :", len(df) - len(fraud_df))

print("\nFraud percentage:")
print(f"{len(fraud_df) / len(df) * 100:.2f}%")

print("\nSaved as:")
print(output_file)

print("\nFirst 5 fraud transactions:")
display(fraud_df.head())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FILE = "fraud.csv"
OUTPUT_FILE = "new_fraud_expanded.csv"
AUDIT_FILE = "synthetic_bridge_audit.csv"

# ------------------------------------------------------------
# Temporal constraint
# ------------------------------------------------------------

# Start conservative.
# 24 hours = 1440 minutes.
MAX_TIME_GAP_MINUTES = 1440

# ------------------------------------------------------------
# Amount constraint
# ------------------------------------------------------------

# Ratio tolerance.
#
# Example:
# 10000 and 500 are acceptable because:
# 10000 / 500 = 20 <= 50
#
# 10000 and 100 are NOT acceptable:
# 10000 / 100 = 100 > 50
#
AMOUNT_RATIO_TOLERANCE = 50


# ------------------------------------------------------------
# Similarity weights
# ------------------------------------------------------------

TIME_WEIGHT = 0.40
BANK_WEIGHT = 0.30
DISTRICT_WEIGHT = 0.30
PAYMENT_MODE_WEIGHT = 0.0

# Minimum score required
MIN_SIMILARITY_SCORE = 0.70


# ------------------------------------------------------------
# Chain constraints
# ------------------------------------------------------------

MIN_CHAIN_LENGTH = 2
MAX_CHAIN_LENGTH = 12


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(INPUT_FILE)

print("=" * 70)
print("LOADING FRAUD DATA")
print("=" * 70)

print("Transactions:", len(df))
print("Columns:", list(df.columns))


# ============================================================
# NORMALIZE
# ============================================================

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

df["from_account"] = (
    df["from_account"]
    .astype(str)
    .str.strip()
)

df["to_account"] = (
    df["to_account"]
    .astype(str)
    .str.strip()
)

df["from_indian_bank"] = (
    df["from_indian_bank"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df["to_indian_bank"] = (
    df["to_indian_bank"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df["from_district"] = (
    df["from_district"]
    .astype(str)
    .str.strip()
)

df["to_district"] = (
    df["to_district"]
    .astype(str)
    .str.strip()
)

df["payment_mode"] = (
    df["payment_mode"]
    .astype(str)
    .str.strip()
    .str.upper()
)

df["amount_paid"] = pd.to_numeric(
    df["amount_paid"],
    errors="coerce"
)

df = df.dropna(
    subset=[
        "timestamp",
        "from_account",
        "to_account",
        "amount_paid"
    ]
).copy()

df = df.sort_values("timestamp").reset_index(drop=True)

# Original transaction ID
df["original_transaction_id"] = np.arange(len(df))

# Mark original transactions
df["is_synthetic_bridge"] = False

print("\nValid transactions:", len(df))


# ============================================================
# ACCOUNT FREQUENCY
# ============================================================

out_count = df["from_account"].value_counts()
in_count = df["to_account"].value_counts()

account_count = (
    out_count
    .add(in_count, fill_value=0)
)

# Accounts appearing only once
unique_accounts = set(
    account_count[
        account_count == 1
    ].index
)

print("\nUnique accounts:", len(unique_accounts))


# ============================================================
# IDENTIFY INDEPENDENT TRANSACTIONS
# ============================================================

# Both endpoints must be one-time accounts.
#
# Example:
#
# A -> B
#
# where A and B do not occur anywhere else.
#
# These are the transactions we primarily want to connect.

independent_mask = (
    df["from_account"].isin(unique_accounts) &
    df["to_account"].isin(unique_accounts)
)

independent_df = df[
    independent_mask
].copy()

print(
    "Independent transactions:",
    len(independent_df)
)

print(
    "Independent percentage:",
    f"{len(independent_df) / len(df) * 100:.2f}%"
)


# ============================================================
# AMOUNT RATIO
# ============================================================

def amount_ratio(amount1, amount2):

    if amount1 <= 0 or amount2 <= 0:
        return np.inf

    return max(amount1, amount2) / min(amount1, amount2)


# ============================================================
# TIME SIMILARITY
# ============================================================

def time_similarity(gap_minutes):

    if gap_minutes < 0:
        return 0.0

    if gap_minutes > MAX_TIME_GAP_MINUTES:
        return 0.0

    # 1.0 = immediately adjacent
    # 0.0 = maximum allowed gap

    return (
        1 -
        gap_minutes / MAX_TIME_GAP_MINUTES
    )


# ============================================================
# CANDIDATE SIMILARITY
# ============================================================

def calculate_similarity(tx1, tx2):

    # --------------------------------------------------------
    # Transaction ordering
    # --------------------------------------------------------

    if tx2["timestamp"] <= tx1["timestamp"]:
        return None

    gap_minutes = (
        tx2["timestamp"] -
        tx1["timestamp"]
    ).total_seconds() / 60

    if gap_minutes > MAX_TIME_GAP_MINUTES:
        return None


    # --------------------------------------------------------
    # Payment mode
    # --------------------------------------------------------

    payment_same = (
        tx1["payment_mode"] ==
        tx2["payment_mode"]
    )

    # We require same payment mode.
    # if not payment_same:
    #     return None


    # --------------------------------------------------------
    # District continuity
    # --------------------------------------------------------

    # Example:
    #
    # A -> B
    # C -> D
    #
    # For B -> C to be created:
    #
    # previous.to_district
    # should correspond to
    # next.from_district

    district_continuity = (
        tx1["to_district"] ==
        tx2["from_district"]
    )

    # if not district_continuity:
    #     return None


    # --------------------------------------------------------
    # Bank continuity
    # --------------------------------------------------------

    bank_continuity = (
        tx1["to_indian_bank"] ==
        tx2["from_indian_bank"]
    )


    # --------------------------------------------------------
    # Amount ratio
    # --------------------------------------------------------

    ratio = amount_ratio(
        tx1["amount_paid"],
        tx2["amount_paid"]
    )

    # if ratio > AMOUNT_RATIO_TOLERANCE:
    #     return None


    # --------------------------------------------------------
    # Score
    # --------------------------------------------------------

    time_score = time_similarity(
        gap_minutes
    )

    payment_score = 1.0

    district_score = 1.0

    bank_score = (
        1.0 if bank_continuity
        else 0.0
    )

    score = (
        TIME_WEIGHT * time_score +
        BANK_WEIGHT * bank_score +
        DISTRICT_WEIGHT * district_score +
        PAYMENT_MODE_WEIGHT * payment_score
    )


    if score < MIN_SIMILARITY_SCORE:
        return None


    return {
        "score": score,
        "gap_minutes": gap_minutes,
        "time_score": time_score,
        "bank_match": bank_continuity,
        "district_match": district_continuity,
        "payment_mode_match": payment_same,
        "amount_ratio": ratio
    }


# ============================================================
# BUILD CANDIDATE EDGES
# ============================================================

print("\n" + "=" * 70)
print("BUILDING CANDIDATE LINKS")
print("=" * 70)

candidates = []

rows = independent_df.to_dict("records")

for i in range(len(rows)):

    tx1 = rows[i]

    for j in range(i + 1, len(rows)):

        tx2 = rows[j]

        # Since sorted by timestamp
        # we can stop once gap becomes too large.

        gap = (
            tx2["timestamp"] -
            tx1["timestamp"]
        ).total_seconds() / 60

        if gap > MAX_TIME_GAP_MINUTES:
            break

        result = calculate_similarity(
            tx1,
            tx2
        )

        if result is None:
            continue

        candidates.append({

            "tx1_id":
                tx1["original_transaction_id"],

            "tx2_id":
                tx2["original_transaction_id"],

            "score":
                result["score"],

            "gap_minutes":
                result["gap_minutes"],

            "time_score":
                result["time_score"],

            "bank_match":
                result["bank_match"],

            "district_match":
                result["district_match"],

            "payment_mode_match":
                result["payment_mode_match"],

            "amount_ratio":
                result["amount_ratio"]
        })


candidate_df = pd.DataFrame(candidates)

print(
    "Candidate links found:",
    len(candidate_df)
)

if len(candidate_df) == 0:

    print("\nNo compatible transactions found.")
    print(
        "Try increasing MAX_TIME_GAP_MINUTES "
        "or lowering MIN_SIMILARITY_SCORE."
    )

    # Still save original data
    df.to_csv(
        OUTPUT_FILE,
        index=False
    )

else:

    candidate_df = candidate_df.sort_values(
        "score",
        ascending=False
    ).reset_index(drop=True)


# ============================================================
# GREEDILY SELECT LINKS
# ============================================================

# Each transaction can have:
#
# maximum 1 predecessor
# maximum 1 successor
#
# Therefore:
#
# A -> B
# C -> D
#
# can become:
#
# A -> B -> C -> D
#
# but B cannot have two successors.

predecessor = {}
successor = {}

selected_links = []


for _, candidate in candidate_df.iterrows():

    tx1 = int(candidate["tx1_id"])
    tx2 = int(candidate["tx2_id"])

    # tx1 already has a successor
    if tx1 in successor:
        continue

    # tx2 already has a predecessor
    if tx2 in predecessor:
        continue

    # Select edge
    successor[tx1] = tx2
    predecessor[tx2] = tx1

    selected_links.append(
        candidate.to_dict()
    )


selected_df = pd.DataFrame(
    selected_links
)

print("\n" + "=" * 70)
print("SELECTED LINKS")
print("=" * 70)

print(
    "Selected synthetic links:",
    len(selected_df)
)


# ============================================================
# BUILD CHAINS
# ============================================================

# Find chain starts:
#
# transaction with successor
# but no predecessor

chain_starts = [
    tx_id
    for tx_id in successor
    if tx_id not in predecessor
]


chains = []

visited = set()


for start in chain_starts:

    chain = []

    current = start

    while current in successor:

        if current in visited:
            break

        visited.add(current)

        chain.append(current)

        current = successor[current]

    # Add final transaction
    if current not in visited:
        chain.append(current)
        visited.add(current)

    if len(chain) >= MIN_CHAIN_LENGTH + 1:
        chains.append(chain)


print("\nChains created:")

chain_lengths = []

for chain in chains:

    length = len(chain)

    chain_lengths.append(length)

    print(
        " -> ".join(
            map(str, chain)
        )
    )


print(
    "\nNumber of chains:",
    len(chains)
)


# ============================================================
# MAP TRANSACTION ID -> ROW
# ============================================================

transaction_lookup = (
    df.set_index(
        "original_transaction_id"
    )
)


# ============================================================
# CREATE SYNTHETIC BRIDGE TRANSACTIONS
# ============================================================

synthetic_rows = []
audit_rows = []

synthetic_id = 1


for _, link in selected_df.iterrows():

    tx1_id = int(link["tx1_id"])
    tx2_id = int(link["tx2_id"])

    tx1 = transaction_lookup.loc[tx1_id]
    tx2 = transaction_lookup.loc[tx2_id]


    # --------------------------------------------------------
    # Synthetic timestamp
    # --------------------------------------------------------

    time1 = tx1["timestamp"]
    time2 = tx2["timestamp"]

    midpoint = (
        time1 +
        (time2 - time1) / 2
    )


    # --------------------------------------------------------
    # Synthetic amount
    # --------------------------------------------------------

    # Geometric mean prevents the bridge amount from
    # simply copying one side.

    amount1 = float(tx1["amount_paid"])
    amount2 = float(tx2["amount_paid"])

    synthetic_amount = np.sqrt(
        amount1 * amount2
    )


    # --------------------------------------------------------
    # Synthetic bridge
    # --------------------------------------------------------

    bridge = {}

    # Preserve all original columns
    for col in df.columns:

        if col in [
            "original_transaction_id",
            "is_synthetic_bridge"
        ]:
            continue

        bridge[col] = np.nan


    # --------------------------------------------------------
    # Create B -> C
    # --------------------------------------------------------

    bridge["timestamp"] = midpoint

    bridge["from_account"] = (
        tx1["to_account"]
    )

    bridge["to_account"] = (
        tx2["from_account"]
    )

    bridge["amount_paid"] = (
        synthetic_amount
    )

    bridge["payment_mode"] = (
        tx1["payment_mode"]
    )

    bridge["is_laundering"] = True


    # --------------------------------------------------------
    # Geographic continuity
    # --------------------------------------------------------

    bridge["from_district"] = (
        tx1["to_district"]
    )

    bridge["to_district"] = (
        tx2["from_district"]
    )


    # --------------------------------------------------------
    # Bank continuity
    # --------------------------------------------------------

    bridge["from_indian_bank"] = (
        tx1["to_indian_bank"]
    )

    bridge["to_indian_bank"] = (
        tx2["from_indian_bank"]
    )


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    bridge["original_transaction_id"] = (
        f"SYNTHETIC_{synthetic_id}"
    )

    bridge["is_synthetic_bridge"] = True

    bridge["synthetic_source_tx1"] = tx1_id
    bridge["synthetic_source_tx2"] = tx2_id

    bridge["synthetic_similarity_score"] = (
        link["score"]
    )

    bridge["synthetic_time_gap_minutes"] = (
        link["gap_minutes"]
    )

    bridge["synthetic_bank_match"] = (
        link["bank_match"]
    )

    bridge["synthetic_district_match"] = (
        link["district_match"]
    )

    bridge["synthetic_amount_ratio"] = (
        link["amount_ratio"]
    )

    synthetic_rows.append(bridge)


    # Audit record

    audit_rows.append({

        "synthetic_id":
            f"SYNTHETIC_{synthetic_id}",

        "previous_transaction":
            tx1_id,

        "next_transaction":
            tx2_id,

        "from_account":
            tx1["to_account"],

        "to_account":
            tx2["from_account"],

        "timestamp":
            midpoint,

        "similarity_score":
            link["score"],

        "time_gap_minutes":
            link["gap_minutes"],

        "bank_match":
            link["bank_match"],

        "district_match":
            link["district_match"],

        "payment_mode_match":
            link["payment_mode_match"],

        "amount_ratio":
            link["amount_ratio"],

        "previous_payment_mode":
            tx1["payment_mode"],

        "next_payment_mode":
            tx2["payment_mode"],

        "previous_to_district":
            tx1["to_district"],

        "next_from_district":
            tx2["from_district"],

        "previous_to_bank":
            tx1["to_indian_bank"],

        "next_from_bank":
            tx2["from_indian_bank"]
    })

    synthetic_id += 1


# ============================================================
# CREATE EXPANDED DATASET
# ============================================================

synthetic_df = pd.DataFrame(
    synthetic_rows
)

# Add missing columns if necessary
for col in df.columns:

    if col not in synthetic_df.columns:
        synthetic_df[col] = np.nan


# Make column ordering consistent
synthetic_df = synthetic_df[
    [
        col for col in df.columns
        if col in synthetic_df.columns
    ]
    +
    [
        col for col in synthetic_df.columns
        if col not in df.columns
    ]
]


expanded_df = pd.concat(
    [
        df,
        synthetic_df
    ],
    ignore_index=True
)


# Sort chronologically
expanded_df = expanded_df.sort_values(
    "timestamp"
).reset_index(drop=True)


# ============================================================
# SAVE
# ============================================================

expanded_df.to_csv(
    OUTPUT_FILE,
    index=False
)

audit_df = pd.DataFrame(
    audit_rows
)

audit_df.to_csv(
    AUDIT_FILE,
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("EXPANSION COMPLETE")
print("=" * 70)

print(
    "Original fraud transactions :",
    len(df)
)

print(
    "Synthetic bridge transactions:",
    len(synthetic_df)
)

print(
    "Expanded transactions        :",
    len(expanded_df)
)

if len(df) > 0:

    print(
        "Dataset increase             :",
        f"{len(synthetic_df) / len(df) * 100:.2f}%"
    )

print(
    "\nSaved:",
    OUTPUT_FILE
)

print(
    "Saved audit:",
    AUDIT_FILE
)

In [ ]:
if len(audit_df) > 0:

    plt.figure(figsize=(10, 5))

    plt.hist(
        audit_df["similarity_score"],
        bins=20
    )

    plt.axvline(
        MIN_SIMILARITY_SCORE,
        linestyle="--",
        label=f"Minimum score = {MIN_SIMILARITY_SCORE}"
    )

    plt.xlabel("Similarity score")
    plt.ylabel("Number of synthetic bridges")
    plt.title("Similarity Scores of Generated Synthetic Bridges")
    plt.legend()
    plt.tight_layout()
    plt.show()

if len(audit_df) > 0:

    plt.figure(figsize=(10, 5))

    plt.hist(
        audit_df["time_gap_minutes"],
        bins=20
    )

    plt.xlabel("Time gap between source transactions (minutes)")
    plt.ylabel("Number of generated bridges")
    plt.title("Temporal Distance Used for Synthetic Chaining")
    plt.tight_layout()
    plt.show()

if len(audit_df) > 0:

    parameter_counts = pd.Series({

        "Same payment mode":
            audit_df["payment_mode_match"].sum(),

        "Same handoff district":
            audit_df["district_match"].sum(),

        "Same handoff bank":
            audit_df["bank_match"].sum(),

    })

    plt.figure(figsize=(9, 5))

    parameter_counts.plot(
        kind="bar"
    )

    plt.ylabel("Number of synthetic bridges")
    plt.xlabel("Similarity condition")
    plt.title(
        "Conditions Satisfied by Generated Synthetic Bridges"
    )

    plt.xticks(
        rotation=20,
        ha="right"
    )

    plt.tight_layout()
    plt.show()

if len(audit_df) > 0:

    plt.figure(figsize=(10, 5))

    plt.hist(
        audit_df["amount_ratio"],
        bins=30
    )

    plt.axvline(
        AMOUNT_RATIO_TOLERANCE,
        linestyle="--",
        label=f"Maximum ratio = {AMOUNT_RATIO_TOLERANCE}x"
    )

    plt.xlabel("Amount ratio: max(amount) / min(amount)")
    plt.ylabel("Number of bridges")
    plt.title("Amount Ratio of Source Transactions Used for Chaining")
    plt.legend()

    plt.tight_layout()
    plt.show()

if len(chain_lengths) > 0:

    chain_counter = Counter(
        chain_lengths
    )

    lengths = sorted(
        chain_counter.keys()
    )

    counts = [
        chain_counter[x]
        for x in lengths
    ]

    plt.figure(figsize=(9, 5))

    plt.bar(
        lengths,
        counts
    )

    plt.xlabel("Number of original transactions in chain")
    plt.ylabel("Number of chains")
    plt.title("Distribution of Generated Fraud Chain Lengths")

    plt.xticks(lengths)

    plt.tight_layout()
    plt.show()

def account_reuse_stats(data):

    out_count = data["from_account"].value_counts()
    in_count = data["to_account"].value_counts()

    total_count = (
        out_count
        .add(in_count, fill_value=0)
    )

    return {
        "1 transaction":
            (total_count == 1).sum(),

        "2 transactions":
            ((total_count >= 2) &
             (total_count <= 2)).sum(),

        "3-5 transactions":
            ((total_count >= 3) &
             (total_count <= 5)).sum(),

        "6-10 transactions":
            ((total_count >= 6) &
             (total_count <= 10)).sum(),

        ">10 transactions":
            (total_count > 10).sum()
    }


before = account_reuse_stats(df)

after = account_reuse_stats(
    expanded_df
)

comparison = pd.DataFrame({
    "Before": before,
    "After": after
})


comparison.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.xlabel("Account transaction involvement")
plt.ylabel("Number of accounts")
plt.title(
    "Account Reuse Before vs After Synthetic Expansion"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ACCOUNT REUSE / DISTRIBUTION ANALYSIS
# remaining_fraud_transactions.csv
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# INPUT_FILE = "remaining_fraud_transactions.csv"
# INPUT_FILE = "still_unpatterned_fraud.csv"
INPUT_FILE = "new_fraud_expanded.csv"

# ------------------------------------------------------------
# 1. LOAD
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

df["from_account"] = (
    df["from_account"]
    .astype(str)
    .str.strip()
)

df["to_account"] = (
    df["to_account"]
    .astype(str)
    .str.strip()
)

df["amount_paid"] = pd.to_numeric(
    df["amount_paid"],
    errors="coerce"
)

print("=" * 70)
print("ACCOUNT REUSE ANALYSIS")
print("=" * 70)

print(f"Total fraud transactions : {len(df):,}")

# ------------------------------------------------------------
# 2. ALL UNIQUE ACCOUNTS
# ------------------------------------------------------------

all_accounts = set(
    df["from_account"]
).union(
    set(df["to_account"])
)

print(f"Unique accounts involved : {len(all_accounts):,}")


# ------------------------------------------------------------
# 3. TRANSACTION COUNT PER ACCOUNT
#
# Count BOTH:
#   outgoing transactions
#   incoming transactions
#   total involvement
# ------------------------------------------------------------

outgoing_count = (
    df.groupby("from_account")
    .size()
    .rename("outgoing_transactions")
)

incoming_count = (
    df.groupby("to_account")
    .size()
    .rename("incoming_transactions")
)

account_stats = pd.DataFrame(
    index=pd.Index(
        list(all_accounts),
        name="account"
    )
)

account_stats = account_stats.join(
    outgoing_count,
    how="left"
)

account_stats = account_stats.join(
    incoming_count,
    how="left"
)

account_stats = account_stats.fillna(0)

account_stats["outgoing_transactions"] = (
    account_stats["outgoing_transactions"]
    .astype(int)
)

account_stats["incoming_transactions"] = (
    account_stats["incoming_transactions"]
    .astype(int)
)

account_stats["total_transactions"] = (
    account_stats["outgoing_transactions"]
    +
    account_stats["incoming_transactions"]
)

# ------------------------------------------------------------
# 4. CLASSIFY ACCOUNT REUSE
# ------------------------------------------------------------

account_stats["account_type"] = np.select(
    [
        (
            (account_stats["outgoing_transactions"] > 0) &
            (account_stats["incoming_transactions"] > 0)
        ),
        (
            account_stats["outgoing_transactions"] > 0
        ),
        (
            account_stats["incoming_transactions"] > 0
        )
    ],
    [
        "Both incoming & outgoing",
        "Outgoing only",
        "Incoming only"
    ],
    default="None"
)

account_stats["reuse_category"] = np.select(
    [
        account_stats["total_transactions"] == 1,
        account_stats["total_transactions"] == 2,
        account_stats["total_transactions"].between(3, 5),
        account_stats["total_transactions"].between(6, 10),
        account_stats["total_transactions"] > 10
    ],
    [
        "1 transaction",
        "2 transactions",
        "3–5 transactions",
        "6–10 transactions",
        ">10 transactions"
    ],
    default="Other"
)


# ------------------------------------------------------------
# 5. PRINT ACCOUNT REUSE STATISTICS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ACCOUNT REUSE")
print("=" * 70)

reuse_distribution = (
    account_stats["reuse_category"]
    .value_counts()
)

print("\nAccounts by number of transaction involvements:")
print(reuse_distribution)


# ------------------------------------------------------------
# 6. HOW MANY TRANSACTIONS COME FROM REUSED ACCOUNTS?
# ------------------------------------------------------------

transaction_account_frequency = (
    account_stats["total_transactions"]
)

# Map frequency to source account
df["from_account_frequency"] = (
    df["from_account"]
    .map(transaction_account_frequency)
)

df["to_account_frequency"] = (
    df["to_account"]
    .map(transaction_account_frequency)
)

# A transaction is associated with a reused account
# if either endpoint participates in >1 transaction.
df["has_reused_account"] = (
    (df["from_account_frequency"] > 1) |
    (df["to_account_frequency"] > 1)
)

reused_transaction_count = (
    df["has_reused_account"].sum()
)

unique_transaction_count = (
    len(df) - reused_transaction_count
)

print("\nTransaction-level result:")
print(
    f"Transactions involving ONLY one-time accounts : "
    f"{unique_transaction_count:,}"
)

print(
    f"Transactions involving a reused account        : "
    f"{reused_transaction_count:,}"
)

print(
    f"Percentage involving reused account            : "
    f"{reused_transaction_count / len(df) * 100:.2f}%"
)


# ============================================================
# VISUALIZATION 1
# Account reuse distribution
# ============================================================

plt.figure(figsize=(10, 6))

order = [
    "1 transaction",
    "2 transactions",
    "3–5 transactions",
    "6–10 transactions",
    ">10 transactions"
]

values = [
    reuse_distribution.get(x, 0)
    for x in order
]

plt.bar(order, values)

plt.title(
    "Distribution of Fraud Accounts by Transaction Involvement"
)

plt.xlabel(
    "Number of transactions involving account"
)

plt.ylabel(
    "Number of accounts"
)

plt.xticks(rotation=20)

plt.tight_layout()
plt.show()


# ============================================================
# VISUALIZATION 2
# Histogram of transaction involvement per account
# ============================================================

plt.figure(figsize=(10, 6))

plt.hist(
    account_stats["total_transactions"],
    bins=30
)

plt.title(
    "Transaction Involvement Distribution per Account"
)

plt.xlabel(
    "Number of transactions involving account"
)

plt.ylabel(
    "Number of accounts"
)

plt.tight_layout()
plt.show()


# ============================================================
# VISUALIZATION 3
# Incoming vs outgoing transactions
# ============================================================

plt.figure(figsize=(10, 6))

plt.scatter(
    account_stats["incoming_transactions"],
    account_stats["outgoing_transactions"],
    alpha=0.6
)

plt.xlabel(
    "Incoming transactions"
)

plt.ylabel(
    "Outgoing transactions"
)

plt.title(
    "Account Activity: Incoming vs Outgoing Transactions"
)

plt.tight_layout()
plt.show()


# ============================================================
# 7. TOP REUSED ACCOUNTS
# ============================================================

print("\n" + "=" * 70)
print("TOP REUSED ACCOUNTS")
print("=" * 70)

top_accounts = (
    account_stats
    .sort_values(
        "total_transactions",
        ascending=False
    )
    .head(30)
)

display(
    top_accounts.reset_index()
)


# ============================================================
# VISUALIZATION 4
# Top 20 most reused accounts
# ============================================================

top20 = (
    account_stats
    .sort_values(
        "total_transactions",
        ascending=False
    )
    .head(20)
    .sort_values(
        "total_transactions"
    )
)

plt.figure(figsize=(10, 8))

plt.barh(
    top20.index.astype(str),
    top20["total_transactions"]
)

plt.title(
    "Top 20 Most Reused Accounts"
)

plt.xlabel(
    "Total transaction involvement"
)

plt.ylabel(
    "Account"
)

plt.tight_layout()
plt.show()


# ============================================================
# 8. ACCOUNT-PAIR REUSE
#
# Does the SAME:
#
#       A -> B
#
# happen repeatedly?
# ============================================================

pair_counts = (
    df.groupby(
        [
            "from_account",
            "to_account"
        ]
    )
    .size()
    .reset_index(
        name="transaction_count"
    )
    .sort_values(
        "transaction_count",
        ascending=False
    )
)

print("\n" + "=" * 70)
print("ACCOUNT-PAIR REUSE")
print("=" * 70)

print(
    f"Unique account pairs : {len(pair_counts):,}"
)

print("\nTop repeated account pairs:")

display(
    pair_counts.head(30)
)


# ============================================================
# VISUALIZATION 5
# Distribution of repeated account pairs
# ============================================================

pair_distribution = (
    pair_counts["transaction_count"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(10, 6))

plt.bar(
    pair_distribution.index,
    pair_distribution.values
)

plt.title(
    "Distribution of Transactions per Account Pair"
)

plt.xlabel(
    "Number of transactions for same A → B pair"
)

plt.ylabel(
    "Number of account pairs"
)

plt.tight_layout()
plt.show()


# ============================================================
# 9. HOW MANY TRANSACTIONS ARE PART OF REPEATED PAIRS?
# ============================================================

repeated_pairs = pair_counts[
    pair_counts["transaction_count"] > 1
]

transactions_in_repeated_pairs = (
    repeated_pairs["transaction_count"]
    .sum()
)

print("\n" + "=" * 70)
print("REPEATED PAIR ANALYSIS")
print("=" * 70)

print(
    f"Repeated account pairs       : "
    f"{len(repeated_pairs):,}"
)

print(
    f"Transactions in repeated pairs: "
    f"{transactions_in_repeated_pairs:,}"
)

print(
    f"Percentage of fraud transactions: "
    f"{transactions_in_repeated_pairs / len(df) * 100:.2f}%"
)


# ============================================================
# 10. REPEATED SOURCE ACCOUNTS
# ============================================================

source_reuse = (
    df.groupby("from_account")
    .size()
    .reset_index(
        name="outgoing_count"
    )
    .sort_values(
        "outgoing_count",
        ascending=False
    )
)

print("\n" + "=" * 70)
print("SOURCE ACCOUNT REUSE")
print("=" * 70)

display(
    source_reuse.head(30)
)


# ============================================================
# 11. REPEATED DESTINATION ACCOUNTS
# ============================================================

destination_reuse = (
    df.groupby("to_account")
    .size()
    .reset_index(
        name="incoming_count"
    )
    .sort_values(
        "incoming_count",
        ascending=False
    )
)

print("\n" + "=" * 70)
print("DESTINATION ACCOUNT REUSE")
print("=" * 70)

display(
    destination_reuse.head(30)
)


# ============================================================
# 12. SAVE ACCOUNT ANALYSIS
# ============================================================

account_stats.to_csv(
    "fraud_account_statistics.csv"
)

pair_counts.to_csv(
    "fraud_account_pair_statistics.csv",
    index=False
)

print("\nSaved:")
print("  ✓ fraud_account_statistics.csv")
print("  ✓ fraud_account_pair_statistics.csv")

In [2]:
# ============================================================
# EXCLUSIVE UNKNOWN PATTERN DISCOVERY
# ============================================================
#
# IMPORTANT:
# A transaction can belong to ONLY ONE discovered pattern.
#
# Algorithm:
#
#   1. Sort fraud transactions chronologically
#   2. Pick earliest unused transaction
#   3. Try to construct a pattern starting from that transaction
#   4. If pattern found:
#        - save pattern
#        - mark ALL its transactions as used
#   5. If no pattern found:
#        - mark seed as processed/unpatterned
#   6. Move to next unused transaction
#
# ============================================================

import pandas as pd
import numpy as np
from collections import defaultdict
from pathlib import Path

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

# INPUT_FILE = "remaining_fraud_transactions.csv"
INPUT_FILE = "new_frauds.csv"

OUTPUT_FILE = "unknown_patterns.txt"
SUMMARY_FILE = "unknown_pattern_summary.csv"
UNPATTERNED_FILE = "still_unpatterned_fraud.csv"

# Temporal constraints
FAN_WINDOW_MINUTES = 1200
CHAIN_WINDOW_MINUTES = 1200

# Structural constraints
MIN_FAN_DEGREE = 2

MIN_CHAIN_LENGTH = 2
MAX_CHAIN_LENGTH = 500

# Amount similarity for chaining
AMOUNT_TOLERANCE = 50.0


# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("=" * 80)
print("LOADING FRAUD TRANSACTIONS")
print("=" * 80)

print(f"Input transactions: {len(df):,}")

# ------------------------------------------------------------
# NORMALIZE
# ------------------------------------------------------------

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

df["from_account"] = (
    df["from_account"]
    .astype(str)
    .str.strip()
)

df["to_account"] = (
    df["to_account"]
    .astype(str)
    .str.strip()
)

df["amount_paid"] = pd.to_numeric(
    df["amount_paid"],
    errors="coerce"
)

df["payment_mode"] = (
    df["payment_mode"]
    .astype(str)
    .str.strip()
)

df["from_district"] = (
    df["from_district"]
    .astype(str)
    .str.strip()
)

df["to_district"] = (
    df["to_district"]
    .astype(str)
    .str.strip()
)

df["from_indian_bank"] = (
    df["from_indian_bank"]
    .astype(str)
    .str.strip()
)

df["to_indian_bank"] = (
    df["to_indian_bank"]
    .astype(str)
    .str.strip()
)

df = df.dropna(
    subset=[
        "timestamp",
        "from_account",
        "to_account",
        "amount_paid"
    ]
).copy()

# ------------------------------------------------------------
# UNIQUE TRANSACTION ID
# ------------------------------------------------------------

df = df.reset_index(drop=True)

df["transaction_id"] = df.index

# Chronological order
df = df.sort_values(
    "timestamp"
).reset_index(drop=True)

print(f"Valid transactions: {len(df):,}")


# ------------------------------------------------------------
# TRANSACTION FORMATTER
# ------------------------------------------------------------

def format_transaction(row):

    return ",".join([
        row["timestamp"].strftime("%Y/%m/%d %H:%M"),
        str(row["from_indian_bank"]),
        str(row["from_account"]),
        str(row["to_indian_bank"]),
        str(row["to_account"]),
        f'{row["amount_paid"]:.2f}',
        str(row["payment_mode"]),
        str(row["is_laundering"]),
        str(row["from_district"]),
        str(row["to_district"])
    ])


# ------------------------------------------------------------
# AMOUNT COMPATIBILITY
# ------------------------------------------------------------

def amount_compatible(a, b):

    if a <= 0 or b <= 0:
        return True

    ratio = b / a

    return (
        1 - AMOUNT_TOLERANCE
        <= ratio
        <=
        1 + AMOUNT_TOLERANCE
    )


# ------------------------------------------------------------
# BUILD INDEXES
# ------------------------------------------------------------

outgoing = defaultdict(list)
incoming = defaultdict(list)

for idx, row in df.iterrows():

    outgoing[
        row["from_account"]
    ].append(idx)

    incoming[
        row["to_account"]
    ].append(idx)


# ------------------------------------------------------------
# IMPORTANT:
# USED TRANSACTION SET
# ------------------------------------------------------------

used_transactions = set()

# Transactions that were examined but didn't form a pattern
unpatterned_transactions = set()

# Final discovered patterns
discovered_patterns = []


# ============================================================
# PATTERN DETECTORS
# ============================================================

# Each detector receives:
#
#   seed_idx
#   available transactions
#
# and returns:
#
#   {
#       "type": ...,
#       "description": ...,
#       "transactions": [...]
#   }
#
# OR None
#
# The detectors ONLY use transactions that are currently
# unused.
# ============================================================


# ------------------------------------------------------------
# 1. FAN-IN
# ------------------------------------------------------------

def find_fan_in(seed_idx, available):

    seed = df.loc[seed_idx]

    target = seed["to_account"]

    # All currently available incoming transactions
    candidates = [
        i for i in incoming.get(target, [])
        if i in available
    ]

    if len(candidates) < MIN_FAN_DEGREE:
        return None

    seed_time = seed["timestamp"]

    valid = []

    for idx in candidates:

        row = df.loc[idx]

        delta = (
            row["timestamp"] - seed_time
        ).total_seconds() / 60

        if abs(delta) <= FAN_WINDOW_MINUTES:

            valid.append(idx)

    # Distinct source accounts
    source_accounts = set(
        df.loc[
            valid,
            "from_account"
        ]
    )

    if len(source_accounts) < MIN_FAN_DEGREE:
        return None

    # Keep chronological transactions
    valid = sorted(
        valid,
        key=lambda i: df.loc[
            i,
            "timestamp"
        ]
    )

    return {
        "type": "FAN-IN",
        "description": (
            f"{len(source_accounts)}-degree Fan-In "
            f"to {target}"
        ),
        "transactions": valid
    }


# ------------------------------------------------------------
# 2. FAN-OUT
# ------------------------------------------------------------

def find_fan_out(seed_idx, available):

    seed = df.loc[seed_idx]

    source = seed["from_account"]

    candidates = [
        i for i in outgoing.get(source, [])
        if i in available
    ]

    if len(candidates) < MIN_FAN_DEGREE:
        return None

    seed_time = seed["timestamp"]

    valid = []

    for idx in candidates:

        row = df.loc[idx]

        delta = (
            row["timestamp"] - seed_time
        ).total_seconds() / 60

        if abs(delta) <= FAN_WINDOW_MINUTES:

            valid.append(idx)

    destination_accounts = set(
        df.loc[
            valid,
            "to_account"
        ]
    )

    if len(destination_accounts) < MIN_FAN_DEGREE:
        return None

    valid = sorted(
        valid,
        key=lambda i: df.loc[
            i,
            "timestamp"
        ]
    )

    return {
        "type": "FAN-OUT",
        "description": (
            f"{len(destination_accounts)}-degree Fan-Out "
            f"from {source}"
        ),
        "transactions": valid
    }


# ------------------------------------------------------------
# 3. GATHER-SCATTER
#
# many -> HUB -> many
# ------------------------------------------------------------

def find_gather_scatter(seed_idx, available):

    seed = df.loc[seed_idx]

    hub = seed["to_account"]

    # Incoming
    in_candidates = [
        i for i in incoming.get(hub, [])
        if i in available
    ]

    if len(in_candidates) < MIN_FAN_DEGREE:
        return None

    seed_time = seed["timestamp"]

    incoming_valid = [
        i for i in in_candidates
        if abs(
            (
                df.loc[i, "timestamp"]
                - seed_time
            ).total_seconds() / 60
        ) <= CHAIN_WINDOW_MINUTES
    ]

    source_accounts = set(
        df.loc[
            incoming_valid,
            "from_account"
        ]
    )

    if len(source_accounts) < MIN_FAN_DEGREE:
        return None

    # Outgoing must happen after incoming
    latest_incoming = max(
        df.loc[
            incoming_valid,
            "timestamp"
        ]
    )

    out_candidates = [
        i for i in outgoing.get(hub, [])
        if i in available
    ]

    outgoing_valid = [
        i for i in out_candidates
        if (
            df.loc[i, "timestamp"] >= latest_incoming
            and
            (
                df.loc[i, "timestamp"]
                - latest_incoming
            ).total_seconds() / 60
            <= CHAIN_WINDOW_MINUTES
        )
    ]

    destination_accounts = set(
        df.loc[
            outgoing_valid,
            "to_account"
        ]
    )

    if len(destination_accounts) < MIN_FAN_DEGREE:
        return None

    transactions = sorted(
        incoming_valid + outgoing_valid,
        key=lambda i: df.loc[
            i,
            "timestamp"
        ]
    )

    return {
        "type": "GATHER-SCATTER",
        "description": (
            f"Gather-Scatter: "
            f"{len(source_accounts)} sources -> "
            f"{hub} -> "
            f"{len(destination_accounts)} destinations"
        ),
        "transactions": transactions
    }


# ------------------------------------------------------------
# 4. SCATTER-GATHER
#
# source -> many intermediates -> destination
# ------------------------------------------------------------

def find_scatter_gather(seed_idx, available):

    seed = df.loc[seed_idx]

    source = seed["from_account"]

    first_hops = [
        i for i in outgoing.get(source, [])
        if i in available
    ]

    seed_time = seed["timestamp"]

    first_hops = [
        i for i in first_hops
        if abs(
            (
                df.loc[i, "timestamp"]
                - seed_time
            ).total_seconds() / 60
        ) <= CHAIN_WINDOW_MINUTES
    ]

    intermediates = set(
        df.loc[
            first_hops,
            "to_account"
        ]
    )

    if len(intermediates) < MIN_FAN_DEGREE:
        return None

    # Find common destination
    destination_map = defaultdict(list)

    for intermediary in intermediates:

        for idx in outgoing.get(
            intermediary,
            []
        ):

            if idx not in available:
                continue

            row = df.loc[idx]

            if row["timestamp"] < seed_time:
                continue

            if (
                row["timestamp"]
                - seed_time
            ).total_seconds() / 60 > CHAIN_WINDOW_MINUTES:
                break

            destination_map[
                row["to_account"]
            ].append(idx)

    best_destination = None
    best_transactions = []

    for destination, indices in destination_map.items():

        sending_intermediates = set(
            df.loc[
                indices,
                "from_account"
            ]
        )

        if len(sending_intermediates) >= MIN_FAN_DEGREE:

            if len(indices) > len(best_transactions):

                best_destination = destination
                best_transactions = indices

    if best_destination is None:
        return None

    transactions = sorted(
        first_hops + best_transactions,
        key=lambda i: df.loc[
            i,
            "timestamp"
        ]
    )

    return {
        "type": "SCATTER-GATHER",
        "description": (
            f"Scatter-Gather: "
            f"{source} -> "
            f"{len(intermediates)} intermediates -> "
            f"{best_destination}"
        ),
        "transactions": transactions
    }


# ------------------------------------------------------------
# 5. MULTI-HOP CHAIN
#
# A -> B -> C -> D
# ------------------------------------------------------------

def find_chain(seed_idx, available):

    seed = df.loc[seed_idx]

    chain = [seed_idx]

    visited = {
        seed["from_account"],
        seed["to_account"]
    }

    current_idx = seed_idx

    while len(chain) < MAX_CHAIN_LENGTH:

        current = df.loc[current_idx]

        next_account = current["to_account"]

        candidates = [
            i for i in outgoing.get(
                next_account,
                []
            )
            if i in available
        ]

        candidates = [
            i for i in candidates
            if i not in chain
        ]

        best = None

        for idx in candidates:

            row = df.loc[idx]

            if row["timestamp"] <= current["timestamp"]:
                continue

            delta = (
                row["timestamp"]
                - current["timestamp"]
            ).total_seconds() / 60

            if delta > CHAIN_WINDOW_MINUTES:
                break

            if row["to_account"] in visited:
                continue

            if not amount_compatible(
                current["amount_paid"],
                row["amount_paid"]
            ):
                continue

            best = idx
            break

        if best is None:
            break

        chain.append(best)

        visited.add(
            df.loc[
                best,
                "to_account"
            ]
        )

        current_idx = best

    if len(chain) < MIN_CHAIN_LENGTH:
        return None

    accounts = [
        df.loc[
            chain[0],
            "from_account"
        ]
    ]

    for idx in chain:

        accounts.append(
            df.loc[
                idx,
                "to_account"
            ]
        )

    return {
        "type": "MULTI-HOP CHAIN",
        "description": (
            f"{len(chain)}-transaction chain: "
            f"{' -> '.join(accounts)}"
        ),
        "transactions": chain
    }


# ------------------------------------------------------------
# 6. CYCLE
#
# A -> B -> C -> A
# ------------------------------------------------------------

def find_cycle(seed_idx, available):

    seed = df.loc[seed_idx]

    start = seed["from_account"]

    current_account = seed["to_account"]

    cycle = [seed_idx]

    visited = {
        start,
        current_account
    }

    current_time = seed["timestamp"]

    while len(cycle) < 5:

        candidates = [
            i for i in outgoing.get(
                current_account,
                []
            )
            if i in available
        ]

        next_idx = None

        for idx in candidates:

            row = df.loc[idx]

            if row["timestamp"] <= current_time:
                continue

            delta = (
                row["timestamp"]
                - current_time
            ).total_seconds() / 60

            if delta > CHAIN_WINDOW_MINUTES:
                break

            destination = row["to_account"]

            # Closing cycle
            if destination == start:

                if len(cycle) >= 2:

                    cycle.append(idx)

                    return {
                        "type": "CYCLE",
                        "description": (
                            f"Cycle detected returning to "
                            f"{start}"
                        ),
                        "transactions": cycle
                    }

            if destination in visited:
                continue

            next_idx = idx
            break

        if next_idx is None:
            break

        cycle.append(next_idx)

        current_account = df.loc[
            next_idx,
            "to_account"
        ]

        current_time = df.loc[
            next_idx,
            "timestamp"
        ]

        visited.add(
            current_account
        )

    return None


# ============================================================
# PATTERN PRIORITY
# ============================================================
#
# More specific structures first.
#
# This is important.
#
# Example:
#
# A -> B
# A -> C
# A -> D
# D -> X
#
# Could be interpreted as Fan-Out OR a more complex structure.
#
# We prefer the more informative structure.
# ============================================================

PATTERN_DETECTORS = [
    ("GATHER-SCATTER", find_gather_scatter),
    ("SCATTER-GATHER", find_scatter_gather),
    ("CYCLE", find_cycle),
    ("FAN-IN", find_fan_in),
    ("FAN-OUT", find_fan_out),
    ("MULTI-HOP CHAIN", find_chain),
]


# ============================================================
# MAIN EXCLUSIVE DISCOVERY LOOP
# ============================================================

print("\n")
print("=" * 80)
print("STARTING EXCLUSIVE PATTERN DISCOVERY")
print("=" * 80)

total = len(df)

for seed_idx in range(total):

    # Already consumed by a previous pattern
    if seed_idx in used_transactions:
        continue

    # Already checked and found nothing
    if seed_idx in unpatterned_transactions:
        continue

    available = set(
        range(total)
    ) - used_transactions

    found_pattern = None

    # --------------------------------------------------------
    # Try every pattern against THIS seed
    # --------------------------------------------------------

    for pattern_name, detector in PATTERN_DETECTORS:

        result = detector(
            seed_idx,
            available
        )

        if result is None:
            continue

        # ----------------------------------------------------
        # Make sure the pattern really contains the seed
        # ----------------------------------------------------

        if seed_idx not in result["transactions"]:
            continue

        # ----------------------------------------------------
        # Safety:
        # no transaction may already be consumed
        # ----------------------------------------------------

        if any(
            idx in used_transactions
            for idx in result["transactions"]
        ):
            continue

        found_pattern = result

        break

    # --------------------------------------------------------
    # Pattern found
    # --------------------------------------------------------

    if found_pattern is not None:

        transactions = found_pattern[
            "transactions"
        ]

        used_transactions.update(
            transactions
        )

        discovered_patterns.append(
            found_pattern
        )

        print(
            f"[PATTERN #{len(discovered_patterns):04d}] "
            f"{found_pattern['type']:18s} | "
            f"{len(transactions):2d} transactions | "
            f"{found_pattern['description']}"
        )

    # --------------------------------------------------------
    # No pattern found
    # --------------------------------------------------------

    else:

        unpatterned_transactions.add(
            seed_idx
        )


# ============================================================
# WRITE UNKNOWN PATTERNS
# ============================================================

print("\n")
print("=" * 80)
print("WRITING unknown_patterns.txt")
print("=" * 80)

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    for number, pattern in enumerate(
        discovered_patterns,
        start=1
    ):

        pattern_type = pattern["type"]
        description = pattern["description"]

        f.write(
            f"BEGIN LAUNDERING ATTEMPT - "
            f"{pattern_type}: {description}\n"
        )

        rows = df.loc[
            pattern["transactions"]
        ].sort_values(
            "timestamp"
        )

        for _, row in rows.iterrows():

            f.write(
                format_transaction(row)
                + "\n"
            )

        f.write(
            f"END LAUNDERING ATTEMPT - "
            f"{pattern_type}: {description}\n\n"
        )


# ============================================================
# SUMMARY
# ============================================================

summary = []

for pattern_type in sorted(
    set(
        p["type"]
        for p in discovered_patterns
    )
):

    matching = [
        p for p in discovered_patterns
        if p["type"] == pattern_type
    ]

    transaction_ids = set()

    for p in matching:

        transaction_ids.update(
            p["transactions"]
        )

    summary.append({
        "pattern_type": pattern_type,
        "pattern_instances": len(matching),
        "transactions_used": len(
            transaction_ids
        )
    })


summary_df = pd.DataFrame(summary)

if len(summary_df) > 0:

    summary_df = summary_df.sort_values(
        "pattern_instances",
        ascending=False
    )

else:

    summary_df = pd.DataFrame(
        columns=[
            "pattern_type",
            "pattern_instances",
            "transactions_used"
        ]
    )

summary_df.to_csv(
    SUMMARY_FILE,
    index=False
)


# ============================================================
# SAVE STILL-UNPATTERNED TRANSACTIONS
# ============================================================

still_unpatterned = df.loc[
    sorted(unpatterned_transactions)
].copy()

still_unpatterned.to_csv(
    UNPATTERNED_FILE,
    index=False
)


# ============================================================
# VALIDATION
# ============================================================

patterned_count = len(
    used_transactions
)

unpatterned_count = len(
    unpatterned_transactions
)

print("\n")
print("=" * 80)
print("DISCOVERY COMPLETE")
print("=" * 80)

print(
    f"Total fraud transactions : {total:,}"
)

print(
    f"Transactions in patterns : {patterned_count:,}"
)

print(
    f"Still unpatterned         : {unpatterned_count:,}"
)

print(
    f"Discovered pattern count  : "
    f"{len(discovered_patterns):,}"
)

print(
    f"\nValidation: "
    f"{patterned_count + unpatterned_count:,} "
    f"/ {total:,}"
)

print("\nPattern summary:")
display(summary_df)

print("\nOutput files:")
print(f"  ✓ {OUTPUT_FILE}")
print(f"  ✓ {SUMMARY_FILE}")
print(f"  ✓ {UNPATTERNED_FILE}")

LOADING FRAUD TRANSACTIONS
Input transactions: 5,406
Valid transactions: 5,406


STARTING EXCLUSIVE PATTERN DISCOVERY
[PATTERN #0001] MULTI-HOP CHAIN    |  2 transactions | 2-transaction chain: 815630C40 -> 815635220 -> 8050C86F0
[PATTERN #0002] FAN-OUT            | 39 transactions | 39-degree Fan-Out from 10042B660
[PATTERN #0003] FAN-OUT            |  6 transactions | 6-degree Fan-Out from 10042B6F0
[PATTERN #0004] FAN-OUT            | 30 transactions | 30-degree Fan-Out from 10042B6A8
[PATTERN #0005] FAN-OUT            |  3 transactions | 3-degree Fan-Out from 10042B978
[PATTERN #0006] MULTI-HOP CHAIN    |  3 transactions | 3-transaction chain: 81A1C7C80 -> 81A1C7D20 -> 81B0CC0B1 -> 81B0CDAB1
[PATTERN #0007] FAN-OUT            |  3 transactions | 3-degree Fan-Out from 10042B7C8
[PATTERN #0008] FAN-OUT            |  4 transactions | 4-degree Fan-Out from 10042B9C0
[PATTERN #0009] MULTI-HOP CHAIN    |  2 transactions | 2-transaction chain: 10042B810 -> 80C8D2440 -> 8135D2770
[PATTERN 

,pattern_type,pattern_instances,transactions_used
4,MULTI-HOP CHAIN,943,4059
2,FAN-OUT,210,1139
1,FAN-IN,59,194
3,GATHER-SCATTER,2,10
0,CYCLE,1,4



Output files:
  ✓ unknown_patterns.txt
  ✓ unknown_pattern_summary.csv
  ✓ still_unpatterned_fraud.csv


In [ ]:
# ============================================================
# SIH 2026 - SECOND STAGE FRAUD EXPANSION
#
# Goal:
#   A -> B
# becomes:
#   A -> B -> F
#
# Every transaction in still_unpatterned_fraud.csv receives
# exactly ONE synthetic successor transaction.
#
# Target accounts are selected from fraud.csv.
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import re
import math
from collections import Counter, defaultdict

# ============================================================
# CONFIGURATION
# ============================================================

FRAUD_FILE = "fraud.csv"

BASE_EXPANDED_FILE = "new_fraud_expanded.csv"

REMAINING_FILE = "still_unpatterned_fraud.csv"

OUTPUT_FILE = "new_fraud_expanded_2.csv"

AUDIT_FILE = "second_stage_synthetic_audit.csv"

DISTANCE_FILE = "telangana_district_distance_matrix.csv"

RANDOM_SEED = 20260907

rng = np.random.default_rng(RANDOM_SEED)


# ------------------------------------------------------------
# Time
# ------------------------------------------------------------

MIN_DELAY_MINUTES = 1

MAX_DELAY_MINUTES = 60


# ------------------------------------------------------------
# Account reuse
# ------------------------------------------------------------

# Preferred target accounts:
#
# frequency = 1
# frequency = 2
#
# If insufficient, the algorithm can progressively relax this.

PREFERRED_MAX_ACCOUNT_FREQUENCY = 2

MAX_ACCOUNT_FREQUENCY = 10


# ------------------------------------------------------------
# Geographic selection
# ------------------------------------------------------------

# Larger value = more geographically spread out.
# Smaller value = stronger preference for nearby districts.

DISTANCE_SCALE_KM = 100.0


# Number of nearest districts considered for target selection.

NEAREST_DISTRICTS = 7


# ------------------------------------------------------------
# Bank preference
# ------------------------------------------------------------

BANK_MATCH_MULTIPLIER = 1.25


# ------------------------------------------------------------
# Amount generation
# ------------------------------------------------------------

# New amount is generated around previous transaction amount.

MIN_AMOUNT_MULTIPLIER = 0.80

MAX_AMOUNT_MULTIPLIER = 1.20


# ============================================================
# COLUMN NORMALIZATION
# ============================================================

def clean_columns(df):

    df = df.copy()

    df.columns = [
        str(c)
        .strip()
        .replace("*", "")
        for c in df.columns
    ]

    return df


# ============================================================
# DISTRICT NAME NORMALIZATION
# ============================================================

def normalize_location_name(x):

    if pd.isna(x):
        return None

    x = str(x).strip()

    x = re.sub(r"\s+", " ", x)

    return x


def canonical_location(x):

    if x is None:
        return None

    x = normalize_location_name(x)

    key = x.lower()

    aliases = {

        "bhadradri-kothagudem":
            "Bhadradri Kothagudem",

        "bhadradri kothagudem":
            "Bhadradri Kothagudem",

        "komaram bheem":
            "Kumuram Bheem",

        "komaram bheem-asifabad":
            "Kumuram Bheem",

        "kumaram bheem":
            "Kumuram Bheem",

        "rajanna-siricilla":
            "Rajanna Sircilla",

        "rajanna siricilla":
            "Rajanna Sircilla",

        "rajanna sircilla":
            "Rajanna Sircilla",

        "warangal (r)":
            "Warangal Rural",

        "warangal rural":
            "Warangal Rural",

        "warangal (u)":
            "Warangal Urban",

        "warangal urban":
            "Warangal Urban",

        "yadadri-bhuvanagiri":
            "Yadadri Bhuvanagiri",

        "yadadri bhuvanagiri":
            "Yadadri Bhuvanagiri",

        "jagtial":
            "Jagtial",

        "jagitial":
            "Jagtial",

        "mahbubnagar":
            "Mahabubnagar",

        "mahbubabad":
            "Mahabubabad",

        "narayanapet":
            "Narayanpet",

        "narayanpet":
            "Narayanpet",

        "medchal":
            "Medchal-Malkajgiri",

        "medchal-malkajgiri":
            "Medchal-Malkajgiri",

    }

    return aliases.get(key, x)


# ============================================================
# LOAD DATA
# ============================================================

fraud = clean_columns(
    pd.read_csv(FRAUD_FILE)
)

remaining = clean_columns(
    pd.read_csv(REMAINING_FILE)
)

base_expanded = clean_columns(
    pd.read_csv(BASE_EXPANDED_FILE)
)


# ============================================================
# NORMALIZE DATA TYPES
# ============================================================

required_columns = [
    "timestamp",
    "from_account",
    "to_account",
    "amount_paid",
    "payment_mode",
    "from_district",
    "to_district",
    "from_indian_bank",
    "to_indian_bank"
]

for name, data in [
    ("fraud", fraud),
    ("remaining", remaining),
    ("base_expanded", base_expanded)
]:

    missing = [
        c for c in required_columns
        if c not in data.columns
    ]

    if missing:

        raise ValueError(
            f"{name} is missing columns: {missing}"
        )


for data in [fraud, remaining, base_expanded]:

    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        errors="coerce"
    )

    data["from_account"] = (
        data["from_account"]
        .astype(str)
        .str.strip()
    )

    data["to_account"] = (
        data["to_account"]
        .astype(str)
        .str.strip()
    )

    data["amount_paid"] = pd.to_numeric(
        data["amount_paid"],
        errors="coerce"
    )

    for col in [
        "from_district",
        "to_district",
        "from_indian_bank",
        "to_indian_bank",
        "payment_mode"
    ]:

        data[col] = (
            data[col]
            .astype(str)
            .str.strip()
        )


    data["from_district_canonical"] = (
        data["from_district"]
        .apply(canonical_location)
    )

    data["to_district_canonical"] = (
        data["to_district"]
        .apply(canonical_location)
    )


# ============================================================
# BASIC SUMMARY
# ============================================================

print("=" * 75)
print("SECOND-STAGE FRAUD EXPANSION")
print("=" * 75)

print(
    "Original fraud transactions       :",
    len(fraud)
)

print(
    "Remaining unpatterned transactions:",
    len(remaining)
)

print(
    "Existing expanded transactions    :",
    len(base_expanded)
)


# ============================================================
# FETCH OFFICIAL TELANGANA DISTRICT HQ DATA
# ============================================================

print("\nFetching Telangana District HQ data...")

GIS_URL = (
    "https://tgrac.telangana.gov.in/"
    "arcgis/rest/services/"
    "EndowmentLands_Folder/"
    "Endowment_Lands_Admin_New/"
    "MapServer/4/query"
)

params = {
    "where": "1=1",
    "outFields": "Dist_Name",
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "json"
}

response = requests.get(
    GIS_URL,
    params=params,
    timeout=30
)

response.raise_for_status()

gis_json = response.json()

if "features" not in gis_json:

    raise RuntimeError(
        "Could not retrieve Telangana district HQ data."
    )


district_records = []


for feature in gis_json["features"]:

    attrs = feature.get(
        "attributes",
        {}
    )

    geometry = feature.get(
        "geometry",
        {}
    )

    name = attrs.get(
        "Dist_Name"
    )

    lat = None
    lon = None

    # Multipoint geometry
    if "points" in geometry:

        points = geometry["points"]

        if points:

            lon = points[0][0]
            lat = points[0][1]

    # Point geometry fallback
    elif (
        "x" in geometry and
        "y" in geometry
    ):

        lon = geometry["x"]
        lat = geometry["y"]


    if (
        name is not None and
        lat is not None and
        lon is not None
    ):

        district_records.append({

            "district":
                canonical_location(name),

            "latitude":
                lat,

            "longitude":
                lon
        })


district_coords = pd.DataFrame(
    district_records
)

district_coords = (
    district_coords
    .drop_duplicates("district")
    .reset_index(drop=True)
)


print(
    "District HQ locations fetched:",
    len(district_coords)
)

display(
    district_coords
)


# ============================================================
# SPECIAL NON-DISTRICT LOCATIONS
# ============================================================
#
# IMPORTANT:
#
# These are NOT silently assigned to districts.
#
# They are representative locations for labels appearing in
# the transaction data that are police/administrative units.
#
# Rachakonda:
# HQ at Neredmet.
#
# Railway Police Secunderabad:
# representative location = Secunderabad Junction.
#
# ============================================================

special_locations = {

    "Rachakonda Commissionerate": (
        17.48299,
        78.54275
    ),

    "Malkajgiri Commissionerate": (
        17.48299,
        78.54275
    ),

    "Railway Police Secunderabad": (
        17.4337,
        78.5016
    )
}


for location, coords in special_locations.items():

    if location not in set(
        district_coords["district"]
    ):

        district_coords.loc[
            len(district_coords)
        ] = {

            "district":
                location,

            "latitude":
                coords[0],

            "longitude":
                coords[1]
        }


# ============================================================
# HAVERSINE DISTANCE
# ============================================================

def haversine_km(
    lat1,
    lon1,
    lat2,
    lon2
):

    R = 6371.0

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    dlat = lat2 - lat1

    dlon = np.radians(lon2 - lon1)

    a = (
        np.sin(dlat / 2) ** 2
        +
        np.cos(lat1)
        *
        np.cos(lat2)
        *
        np.sin(dlon / 2) ** 2
    )

    return (
        2 *
        R *
        np.arcsin(
            np.sqrt(a)
        )
    )


# ============================================================
# CREATE DISTANCE MATRIX
# ============================================================

locations = (
    district_coords["district"]
    .tolist()
)

distance_matrix = pd.DataFrame(
    index=locations,
    columns=locations,
    dtype=float
)


for loc1 in locations:

    lat1 = float(
        district_coords.loc[
            district_coords["district"] == loc1,
            "latitude"
        ].iloc[0]
    )

    lon1 = float(
        district_coords.loc[
            district_coords["district"] == loc1,
            "longitude"
        ].iloc[0]
    )

    for loc2 in locations:

        lat2 = float(
            district_coords.loc[
                district_coords["district"] == loc2,
                "latitude"
            ].iloc[0]
        )

        lon2 = float(
            district_coords.loc[
                district_coords["district"] == loc2,
                "longitude"
            ].iloc[0]
        )

        distance_matrix.loc[
            loc1,
            loc2
        ] = haversine_km(
            lat1,
            lon1,
            lat2,
            lon2
        )


distance_matrix.to_csv(
    DISTANCE_FILE
)

print(
    "\nDistance matrix saved:",
    DISTANCE_FILE
)


# ============================================================
# VISUAL 1 — DISTANCE MATRIX
# ============================================================

plt.figure(
    figsize=(18, 14)
)

sns.heatmap(
    distance_matrix,
    cmap="viridis",
    square=True
)

plt.title(
    "Telangana District / Location HQ Distance Matrix"
)

plt.xlabel(
    "Destination district / location"
)

plt.ylabel(
    "Source district / location"
)

plt.tight_layout()

plt.show()


# ============================================================
# BUILD ACCOUNT PROFILES FROM ORIGINAL fraud.csv
# ============================================================
#
# VERY IMPORTANT:
#
# We never invent an account's district.
#
# For every account we inspect its actual appearances:
#
# from_account -> from_district
# to_account   -> to_district
#
# Every observed account/location/bank combination becomes
# an observed profile.
# ============================================================

profile_records = []


# From-account observations

for _, row in fraud.iterrows():

    profile_records.append({

        "account":
            row["from_account"],

        "district":
            row["from_district_canonical"],

        "bank":
            row["from_indian_bank"],

        "payment_mode":
            row["payment_mode"]
    })


# To-account observations

for _, row in fraud.iterrows():

    profile_records.append({

        "account":
            row["to_account"],

        "district":
            row["to_district_canonical"],

        "bank":
            row["to_indian_bank"],

        "payment_mode":
            row["payment_mode"]
    })


profiles = pd.DataFrame(
    profile_records
)


# Remove invalid observations

profiles = profiles.dropna(
    subset=[
        "account",
        "district",
        "bank"
    ]
)


# ============================================================
# ACCOUNT TOTAL FREQUENCY
# ============================================================

account_frequency = Counter()

for account in fraud["from_account"]:
    account_frequency[account] += 1

for account in fraud["to_account"]:
    account_frequency[account] += 1


# ============================================================
# PROFILE FREQUENCY
# ============================================================

profile_counts = (
    profiles
    .groupby(
        [
            "account",
            "district",
            "bank"
        ]
    )
    .size()
    .reset_index(
        name="profile_frequency"
    )
)


profile_counts["account_frequency"] = (
    profile_counts["account"]
    .map(account_frequency)
)


# ============================================================
# EXCLUDE ACCOUNTS THAT ARE PART OF REMAINING TRANSACTIONS
# ============================================================
#
# This is important.
#
# Suppose:
#
# A -> B
# C -> D
#
# is remaining.
#
# We don't want to choose C as the synthetic target for B,
# because that would accidentally create:
#
# A -> B -> C -> D
#
# and merge two transactions that were supposed to receive
# independent synthetic successors.
# ============================================================

remaining_accounts = set(
    remaining["from_account"]
).union(
    set(
        remaining["to_account"]
    )
)


candidate_profiles = profile_counts[
    ~profile_counts["account"].isin(
        remaining_accounts
    )
].copy()


print(
    "\nCandidate account profiles:",
    len(candidate_profiles)
)


# ============================================================
# CANDIDATE ACCOUNT FREQUENCY DISTRIBUTION
# ============================================================

candidate_frequency_counts = (
    candidate_profiles[
        "account_frequency"
    ]
    .value_counts()
    .sort_index()
)


print(
    "\nCandidate account frequency distribution:"
)

print(
    candidate_frequency_counts
)


# ============================================================
# LOCATION DISTANCE LOOKUP
# ============================================================

distance_locations = set(
    distance_matrix.index
)


# ============================================================
# CHOOSE TARGET ACCOUNT
# ============================================================

def choose_target_profile(
    previous_district,
    previous_bank,
    previous_account,
    used_synthetic_targets
):

    previous_district = canonical_location(
        previous_district
    )

    # --------------------------------------------------------
    # Remove impossible candidates
    # --------------------------------------------------------

    candidates = candidate_profiles[
        candidate_profiles["account"] != previous_account
    ].copy()


    if len(candidates) == 0:

        raise RuntimeError(
            "No candidate accounts available."
        )


    # --------------------------------------------------------
    # Require known geographic location
    # --------------------------------------------------------

    candidates = candidates[
        candidates["district"].isin(
            distance_locations
        )
    ].copy()


    # --------------------------------------------------------
    # Prefer low-frequency accounts
    # --------------------------------------------------------

    preferred = candidates[
        candidates["account_frequency"]
        <= PREFERRED_MAX_ACCOUNT_FREQUENCY
    ].copy()


    if len(preferred) >= 10:

        candidates = preferred


    else:

        relaxed = candidates[
            candidates["account_frequency"]
            <= MAX_ACCOUNT_FREQUENCY
        ].copy()

        if len(relaxed) > 0:

            candidates = relaxed


    # --------------------------------------------------------
    # Previous location must be known
    # --------------------------------------------------------

    if previous_district not in distance_matrix.index:

        # Cannot fabricate a district.
        #
        # Use only accounts with same observed location label.

        same_location = candidates[
            candidates["district"] ==
            previous_district
        ].copy()

        if len(same_location) > 0:

            candidates = same_location

        else:

            # Last-resort geographic choice:
            # choose globally lowest-frequency profiles.
            #
            # No district is assigned to the account.
            # Its original observed district is preserved.

            candidates = candidates.sort_values(
                [
                    "account_frequency",
                    "profile_frequency"
                ]
            ).head(100)

    else:

        # ----------------------------------------------------
        # Calculate geographic distance
        # ----------------------------------------------------

        candidates["distance_km"] = (
            candidates["district"]
            .apply(
                lambda x:
                distance_matrix.loc[
                    previous_district,
                    x
                ]
            )
        )


        # ----------------------------------------------------
        # Keep nearest districts
        # ----------------------------------------------------

        nearest_districts = (
            candidates[
                "district"
            ]
            .drop_duplicates()
            .sort_values(
                key=lambda s:
                s.map(
                    lambda x:
                    distance_matrix.loc[
                        previous_district,
                        x
                    ]
                )
            )
            .head(
                NEAREST_DISTRICTS
            )
            .tolist()
        )


        candidates = candidates[
            candidates["district"].isin(
                nearest_districts
            )
        ].copy()


    # --------------------------------------------------------
    # Avoid repeatedly using same synthetic target
    # --------------------------------------------------------

    candidates["synthetic_use_count"] = (
        candidates["account"]
        .map(
            used_synthetic_targets
        )
        .fillna(0)
    )


    # --------------------------------------------------------
    # Distance score
    # --------------------------------------------------------

    if "distance_km" not in candidates.columns:

        candidates["distance_km"] = 0.0


    candidates["distance_score"] = np.exp(
        -candidates["distance_km"]
        / DISTANCE_SCALE_KM
    )


    # --------------------------------------------------------
    # Low-reuse score
    # --------------------------------------------------------

    candidates["reuse_score"] = (
        1 /
        np.sqrt(
            candidates["account_frequency"]
            + 1
        )
    )


    # --------------------------------------------------------
    # Synthetic reuse penalty
    # --------------------------------------------------------

    candidates["target_reuse_penalty"] = (
        1 /
        (
            1 +
            candidates["synthetic_use_count"]
        )
    )


    # --------------------------------------------------------
    # Bank similarity
    # --------------------------------------------------------

    candidates["bank_score"] = np.where(

        candidates["bank"] ==
        previous_bank,

        BANK_MATCH_MULTIPLIER,

        1.0
    )


    # --------------------------------------------------------
    # FINAL SCORE
    # --------------------------------------------------------

    candidates["selection_score"] = (

        candidates["distance_score"]

        *

        candidates["reuse_score"]

        *

        candidates["target_reuse_penalty"]

        *

        candidates["bank_score"]
    )


    # --------------------------------------------------------
    # Top candidate pool
    # --------------------------------------------------------

    candidates = candidates.sort_values(
        "selection_score",
        ascending=False
    )


    top_n = min(
        25,
        len(candidates)
    )

    candidates = candidates.head(
        top_n
    ).copy()


    # --------------------------------------------------------
    # Weighted random selection
    # --------------------------------------------------------
    #
    # This is NOT random district assignment.
    #
    # Randomness happens only among already-valid candidate
    # accounts/profiles after geographic and reuse scoring.
    # --------------------------------------------------------

    weights = (
        candidates["selection_score"]
        .to_numpy()
    )

    weights = np.maximum(
        weights,
        1e-12
    )

    weights = (
        weights /
        weights.sum()
    )


    selected_idx = rng.choice(
        len(candidates),
        p=weights
    )


    selected = candidates.iloc[
        selected_idx
    ]


    return selected


# ============================================================
# GENERATE ONE SUCCESSOR FOR EVERY REMAINING TRANSACTION
# ============================================================

synthetic_rows = []

audit_rows = []

used_synthetic_targets = Counter()


for idx, row in remaining.iterrows():

    previous_account = (
        row["to_account"]
    )

    previous_district = (
        row["to_district_canonical"]
    )

    previous_bank = (
        row["to_indian_bank"]
    )

    previous_amount = float(
        row["amount_paid"]
    )


    # --------------------------------------------------------
    # Select target account
    # --------------------------------------------------------

    target = choose_target_profile(

        previous_district=
            previous_district,

        previous_bank=
            previous_bank,

        previous_account=
            previous_account,

        used_synthetic_targets=
            used_synthetic_targets
    )


    target_account = (
        target["account"]
    )

    target_district = (
        target["district"]
    )

    target_bank = (
        target["bank"]
    )


    # --------------------------------------------------------
    # RANDOM DELAY
    # --------------------------------------------------------

    delay_minutes = int(
        rng.integers(
            MIN_DELAY_MINUTES,
            MAX_DELAY_MINUTES + 1
        )
    )


    new_timestamp = (
        row["timestamp"]
        +
        pd.Timedelta(
            minutes=delay_minutes
        )
    )


    # --------------------------------------------------------
    # RANDOM AMOUNT
    # --------------------------------------------------------

    amount_multiplier = rng.uniform(
        MIN_AMOUNT_MULTIPLIER,
        MAX_AMOUNT_MULTIPLIER
    )

    new_amount = (
        previous_amount *
        amount_multiplier
    )


    # --------------------------------------------------------
    # Create synthetic transaction
    # --------------------------------------------------------
    #
    # B -> F
    #
    # B = previous to_account
    # F = selected existing account
    #
    # IMPORTANT:
    #
    # B's district/bank come directly from the previous
    # transaction.
    #
    # F's district/bank come from fraud.csv.
    # --------------------------------------------------------

    synthetic = {}

    for col in base_expanded.columns:

        synthetic[col] = np.nan


    synthetic["timestamp"] = (
        new_timestamp
    )

    synthetic["from_account"] = (
        previous_account
    )

    synthetic["to_account"] = (
        target_account
    )

    synthetic["amount_paid"] = (
        round(
            new_amount,
            2
        )
    )

    synthetic["payment_mode"] = (
        row["payment_mode"]
    )

    synthetic["is_laundering"] = True


    # STRICT:
    # Previous account's actual observed location

    synthetic["from_district"] = (
        row["to_district"]
    )

    # STRICT:
    # Target account's actual observed location

    synthetic["to_district"] = (
        target_district
    )


    # STRICT bank handling

    synthetic["from_indian_bank"] = (
        row["to_indian_bank"]
    )

    synthetic["to_indian_bank"] = (
        target_bank
    )


    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    synthetic["original_transaction_id"] = (
        f"SECOND_STAGE_SYNTH_{idx}"
    )

    synthetic["is_synthetic_bridge"] = True

    synthetic["synthetic_source_tx1"] = (
        row.get(
            "transaction_id",
            idx
        )
    )

    synthetic["synthetic_source_tx2"] = (
        ""
    )

    synthetic["synthetic_similarity_score"] = (
        target["selection_score"]
    )

    synthetic["synthetic_time_gap_minutes"] = (
        delay_minutes
    )

    synthetic["synthetic_bank_match"] = (
        target_bank ==
        previous_bank
    )

    synthetic["synthetic_district_match"] = (
        target_district ==
        previous_district
    )

    synthetic["synthetic_amount_ratio"] = (
        max(
            previous_amount,
            new_amount
        )
        /
        min(
            previous_amount,
            new_amount
        )
    )


    # Additional metadata

    synthetic[
        "second_stage_generation"
    ] = True

    synthetic[
        "target_account_original_frequency"
    ] = target[
        "account_frequency"
    ]

    synthetic[
        "target_account_profile_frequency"
    ] = target[
        "profile_frequency"
    ]

    synthetic[
        "target_district_distance_km"
    ] = target[
        "distance_km"
    ]

    synthetic[
        "target_selection_score"
    ] = target[
        "selection_score"
    ]

    synthetic[
        "synthetic_delay_minutes"
    ] = delay_minutes


    synthetic_rows.append(
        synthetic
    )


    # --------------------------------------------------------
    # Audit
    # --------------------------------------------------------

    audit_rows.append({

        "remaining_transaction_index":
            idx,

        "previous_account":
            previous_account,

        "previous_district":
            previous_district,

        "previous_bank":
            previous_bank,

        "target_account":
            target_account,

        "target_district":
            target_district,

        "target_bank":
            target_bank,

        "target_original_frequency":
            target[
                "account_frequency"
            ],

        "target_profile_frequency":
            target[
                "profile_frequency"
            ],

        "distance_km":
            target[
                "distance_km"
            ],

        "selection_score":
            target[
                "selection_score"
            ],

        "bank_match":
            target_bank ==
            previous_bank,

        "same_district":
            target_district ==
            previous_district,

        "delay_minutes":
            delay_minutes,

        "amount_multiplier":
            amount_multiplier,

        "previous_amount":
            previous_amount,

        "synthetic_amount":
            new_amount,

        "payment_mode":
            row["payment_mode"]
    })


    used_synthetic_targets[
        target_account
    ] += 1


# ============================================================
# CONVERT TO DATAFRAME
# ============================================================

synthetic_df = pd.DataFrame(
    synthetic_rows
)

audit_df = pd.DataFrame(
    audit_rows
)


# ============================================================
# ADD MISSING COLUMNS
# ============================================================

for col in base_expanded.columns:

    if col not in synthetic_df.columns:

        synthetic_df[col] = np.nan


# ============================================================
# COMBINE WITH EXISTING EXPANDED DATASET
# ============================================================

synthetic_df = synthetic_df[
    [
        c for c in base_expanded.columns
    ]
    +
    [
        c for c in synthetic_df.columns
        if c not in base_expanded.columns
    ]
]


final_df = pd.concat(
    [
        base_expanded,
        synthetic_df
    ],
    ignore_index=True
)


# ============================================================
# SORT BY TIME
# ============================================================

final_df = (
    final_df
    .sort_values(
        "timestamp"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# SAVE FINAL DATASET
# ============================================================

final_df.to_csv(
    OUTPUT_FILE,
    index=False
)

audit_df.to_csv(
    AUDIT_FILE,
    index=False
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 75)
print("SECOND-STAGE EXPANSION COMPLETE")
print("=" * 75)

print(
    "Original fraud transactions          :",
    len(fraud)
)

print(
    "Previously expanded transactions     :",
    len(base_expanded)
)

print(
    "Remaining transactions               :",
    len(remaining)
)

print(
    "New synthetic successor transactions:",
    len(synthetic_df)
)

print(
    "Final transactions                   :",
    len(final_df)
)

print(
    "Expected new transactions            :",
    len(remaining)
)

print(
    "Coverage                             :",
    f"{len(synthetic_df) / len(remaining) * 100:.2f}%"
)


# ============================================================
# VERIFY EXACTLY ONE SUCCESSOR PER
# REMAINING TRANSACTION
# ============================================================

print("\n" + "=" * 75)
print("COVERAGE VERIFICATION")
print("=" * 75)

print(
    "Remaining transactions:",
    len(remaining)
)

print(
    "Generated successors:",
    len(synthetic_df)
)

if len(synthetic_df) == len(remaining):

    print(
        "STATUS: PASS — every remaining transaction "
        "received exactly one successor."
    )

else:

    print(
        "STATUS: FAIL — successor count does not match."
    )


# ============================================================
# VERIFY TIME DELAYS
# ============================================================

print("\n" + "=" * 75)
print("TIME DELAY VERIFICATION")
print("=" * 75)

print(
    "Minimum delay:",
    audit_df["delay_minutes"].min(),
    "minutes"
)

print(
    "Maximum delay:",
    audit_df["delay_minutes"].max(),
    "minutes"
)

print(
    "Average delay:",
    round(
        audit_df["delay_minutes"].mean(),
        2
    ),
    "minutes"
)


# ============================================================
# VERIFY ACCOUNT DISTRICTS
# ============================================================

print("\n" + "=" * 75)
print("DISTRICT INTEGRITY CHECK")
print("=" * 75)

district_failures = []

for _, row in audit_df.iterrows():

    target_account = (
        row["target_account"]
    )

    target_district = (
        row["target_district"]
    )

    observed = profiles[
        profiles["account"] ==
        target_account
    ]["district"].unique()

    if target_district not in observed:

        district_failures.append(
            (
                target_account,
                target_district
            )
        )


print(
    "District integrity failures:",
    len(district_failures)
)

if len(district_failures) == 0:

    print(
        "STATUS: PASS — no synthetic target was "
        "assigned an invented district."
    )

else:

    print(
        "STATUS: FAIL"
    )


# ============================================================
# SUMMARY OF TARGET ACCOUNT REUSE
# ============================================================

target_usage = (
    audit_df[
        "target_account"
    ]
    .value_counts()
)

print("\n" + "=" * 75)
print("SYNTHETIC TARGET ACCOUNT REUSE")
print("=" * 75)

print(
    target_usage.describe()
)

print(
    "\nTop synthetic target accounts:"
)

display(
    target_usage
    .head(20)
    .rename(
        "synthetic_successor_count"
    )
)


# ============================================================
# VISUAL 2 — TARGET ACCOUNT FREQUENCY
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.hist(
    audit_df[
        "target_original_frequency"
    ],
    bins=20
)

plt.xlabel(
    "Original fraud transaction involvement"
)

plt.ylabel(
    "Number of synthetic transactions"
)

plt.title(
    "Reuse Frequency of Accounts Selected as Synthetic Targets"
)

plt.tight_layout()

plt.show()


# ============================================================
# VISUAL 3 — DISTANCE DISTRIBUTION
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.hist(
    audit_df["distance_km"],
    bins=25
)

plt.xlabel(
    "Distance from previous transaction's district HQ (km)"
)

plt.ylabel(
    "Number of synthetic transactions"
)

plt.title(
    "District Distance Used for Synthetic Successor Selection"
)

plt.tight_layout()

plt.show()


# ============================================================
# VISUAL 4 — DELAY DISTRIBUTION
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.hist(
    audit_df["delay_minutes"],
    bins=30
)

plt.xlabel(
    "Random delay before synthetic transaction (minutes)"
)

plt.ylabel(
    "Number of synthetic transactions"
)

plt.title(
    "Random Time Delay Distribution"
)

plt.axvline(
    60,
    linestyle="--",
    label="Maximum allowed = 60 minutes"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# VISUAL 5 — SAME DISTRICT VS MOVED DISTRICT
# ============================================================

district_counts = pd.Series({

    "Same district":
        audit_df["same_district"].sum(),

    "Different district":
        (~audit_df["same_district"]).sum()
})


plt.figure(
    figsize=(8, 5)
)

district_counts.plot(
    kind="bar"
)

plt.ylabel(
    "Synthetic transactions"
)

plt.title(
    "Geographic Movement of Synthetic Successor Transactions"
)

plt.xticks(
    rotation=0
)

plt.tight_layout()

plt.show()


# ============================================================
# VISUAL 6 — BANK MATCH
# ============================================================

bank_counts = pd.Series({

    "Same bank":
        audit_df["bank_match"].sum(),

    "Different bank":
        (~audit_df["bank_match"]).sum()
})


plt.figure(
    figsize=(8, 5)
)

bank_counts.plot(
    kind="bar"
)

plt.ylabel(
    "Synthetic transactions"
)

plt.title(
    "Bank Continuity of Synthetic Successor Transactions"
)

plt.xticks(
    rotation=0
)

plt.tight_layout()

plt.show()


# ============================================================
# VISUAL 7 — DISTANCE VS ACCOUNT REUSE
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.scatter(
    audit_df[
        "distance_km"
    ],
    audit_df[
        "target_original_frequency"
    ],
    alpha=0.6
)

plt.xlabel(
    "District distance (km)"
)

plt.ylabel(
    "Target account's original reuse frequency"
)

plt.title(
    "Geographic Distance vs Target Account Reuse"
)

plt.tight_layout()

plt.show()


# ============================================================
# VISUAL 8 — DISTRICT TRANSITION MATRIX
# ============================================================

transition_matrix = pd.crosstab(

    audit_df[
        "previous_district"
    ],

    audit_df[
        "target_district"
    ]
)


plt.figure(
    figsize=(16, 12)
)

sns.heatmap(
    transition_matrix,
    cmap="Blues"
)

plt.title(
    "Synthetic Fraud Successor District Transitions"
)

plt.xlabel(
    "Target account district"
)

plt.ylabel(
    "Previous transaction district"
)

plt.tight_layout()

plt.show()


# ============================================================
# VISUAL 9 — BEFORE / AFTER ACCOUNT REUSE
# ============================================================

def account_reuse_distribution(data):

    out_counts = (
        data["from_account"]
        .value_counts()
    )

    in_counts = (
        data["to_account"]
        .value_counts()
    )

    total = (
        out_counts
        .add(
            in_counts,
            fill_value=0
        )
    )

    return pd.Series({

        "1 transaction":
            (total == 1).sum(),

        "2 transactions":
            (total == 2).sum(),

        "3-5 transactions":
            (
                (total >= 3) &
                (total <= 5)
            ).sum(),

        "6-10 transactions":
            (
                (total >= 6) &
                (total <= 10)
            ).sum(),

        ">10 transactions":
            (total > 10).sum()
    })


before = account_reuse_distribution(
    base_expanded
)

after = account_reuse_distribution(
    final_df
)


comparison = pd.DataFrame({

    "Before second-stage":
        before,

    "After second-stage":
        after
})


comparison.plot(
    kind="bar",
    figsize=(11, 6)
)

plt.xlabel(
    "Account transaction involvement"
)

plt.ylabel(
    "Number of accounts"
)

plt.title(
    "Account Reuse Before vs After Second-Stage Expansion"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()

plt.show()


# ============================================================
# FINAL FILES
# ============================================================

print("\n" + "=" * 75)
print("FILES CREATED")
print("=" * 75)

print(
    OUTPUT_FILE
)

print(
    AUDIT_FILE
)

print(
    DISTANCE_FILE
)